<a href="https://colab.research.google.com/github/ShaneHurley/BasketballElo/blob/main/nba_unified_prediction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Score & Spread Prediction — Unified Hierarchical + xPPP Pipeline
# Combines: Hierarchical Possession Engine + xPoints Elo Tracker + XGBoost MetaModel

In [ ]:
pip install optuna pandas numpy scikit-learn xgboost tqdm nba_api catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
import subprocess, sys

REQUIRED = ["optuna", "pandas", "numpy", "scikit-learn", "xgboost",
            "nbainjuries", "tqdm"]
for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import warnings
warnings.filterwarnings("ignore")

import math, itertools, datetime as dt
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import optuna
from tqdm.auto import tqdm
import gc

from sklearn.metrics import mean_absolute_error, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("✅ All imports OK.")


✅ All imports OK.


## Cell 2 — Configuration, Paths, Team Maps

In [ ]:
# ─── Google Colab vs. Local ────────────────────────────────────────
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    # Using the standard Colab Drive mount path
    drive.mount('/content/drive')
    ROOT = Path("/content/drive/MyDrive/basketballData")
else:
    ROOT = Path("./basketballData")

ROOT.mkdir(parents=True, exist_ok=True)

# ─── Data Paths ──────────────────────────────────────────────────────
# V3-format seasons (21-22, 22-23, 23-24, 24-25)  ← used for engine training
V3_DATA_PATHS = {
    2021: ROOT / "events_2021_22_pbp_V3.csv",
    2022: ROOT / "events_2022_23_pbp_V3.csv",
    2023: ROOT / "events_2023_24_pbp_V3.csv",
    2024: ROOT / "events_2024_25_pbp_V3.csv",
}

# 2025-26 combined-stats format  ← used for walk-forward test
PBP_2026_PATH    = ROOT / "[10-21-2025]-[05-18-2026]-combined-stats.csv"

# ─── Modern Betting Locations & Names ────────────────────────────────
SPREAD_CSV_PATH  = ROOT / "nba_betting_spread.csv"
ML_CSV_PATH      = ROOT / "nba_betting_money_line.csv"
MODERN_ODDS_PATH = ROOT / "all_odds.csv"
ODDS_CSV_PATH    = MODERN_ODDS_PATH  # Fallback just in case older code calls it

# ─── Reference Data Paths ────────────────────────────────────────────
PLAYER_LIST_PATH = ROOT / "nba_players_all.csv"

# ─── Constants ───────────────────────────────────────────────────────
BASE_ELO              = 1500.0
DEFAULT_LEAGUE_RTG    = 110.0      # points-per-100 used by HierarchicalEngine
DEFAULT_LEAGUE_XPPP   = 1.10       # expected points per possession
OFFSEASON_REVERSION   = 0.15
ASSIST_SPLIT          = 0.45
USAGE_FLOOR           = 0.15
ALTITUDE_TEAMS        = {"DEN", "UTA"}
GOOD_BET_EDGE         = 2.5        # minimum point-edge for ATS recommendation

# ─── Team name → abbreviation ────────────────────────────────────────
TEAM_MAP = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Clippers": "LAC", "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM", "Miami Heat": "MIA", "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN", "New Orleans Pelicans": "NOP", "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC", "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX", "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS", "Toronto Raptors": "TOR", "Utah Jazz": "UTA",
    "Washington Wizards": "WAS",
}

# ─── Player name lookup ──────────────────────────────────────────────
names_dict  = {}   # id -> full_name
name_to_id  = {}   # full_name -> id

if PLAYER_LIST_PATH.exists():
    _names_df  = pd.read_csv(PLAYER_LIST_PATH, low_memory=False)

    # Safely handle both legacy and modern CSV column cases
    pid_col  = "person_id" if "person_id" in _names_df.columns else "PERSON_ID"
    name_col = "display_first_last" if "display_first_last" in _names_df.columns else "DISPLAY_FIRST_LAST"

    names_dict = _names_df.set_index(pid_col)[name_col].to_dict()
    name_to_id = {v: k for k, v in names_dict.items()}
    print(f"✅ Loaded {len(names_dict):,} player names.")
else:
    print("⚠️  PlayerList CSV not found — player-name resolution disabled.")

print("✅ Config ready.")

Mounted at /content/drive
✅ Loaded 4,885 player names.
✅ Config ready.


## Cell 3 — PBP Standardization: V3 & 2025-26 formats + xPoints Calculation

In [ ]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────
def _coerce_int(x):
    if x is None or (isinstance(x, float) and math.isnan(x)): return None
    try: return int(float(str(x).strip()))
    except Exception: return None

def _parse_player_string(s):
    """
    Parses lineup string entries. Handles both classic hyphenated/spaced IDs
    and modern string name lists (e.g., 'Cason Wallace, Chet Holmgren').
    """
    if pd.isna(s) or str(s).strip() == "" or str(s).lower() == "nan":
        return []

    s_str = str(s).strip()

    # Check if this is the modern 2026 text format (contains alpha names)
    if any(c.isalpha() for c in s_str) or "," in s_str:
        # Split by comma or dashes if used as delimiters
        tokens = [t.strip() for t in s_str.replace("-", ",").split(",") if t.strip()]
        processed_ids = []
        for token in tokens:
            if token.lower() == "nan":
                continue
            # Check if name_to_id map exists in global namespace, else use stable string hashing
            if 'name_to_id' in globals() and token in name_to_id:
                processed_ids.append(int(name_to_id[token]))
            else:
                # Generate a stable, positive integer ID from the player's name string
                stable_id = abs(hash(token)) % 1000000
                processed_ids.append(stable_id)
        return processed_ids
    else:
        # Traditional numeric ID string format (e.g., '201939 - 203507')
        clean_str = s_str.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        delimiters = ["-", ",", " "]
        for d in delimiters:
            if d in clean_str:
                return [int(float(t.strip())) for t in clean_str.split(d) if t.strip() and t.lower() != "nan"]
        # Single element fallback
        try:
            return [int(float(clean_str))]
        except ValueError:
            return []

def _map_player_name(raw, name_to_id):
    """Resolve 'id/Name' or 'Full Name' to integer player-id."""
    if pd.isna(raw) or not str(raw).strip(): return None
    s = str(raw).strip()
    if "/" in s:
        try: return int(s.split("/")[0])
        except ValueError: s = s.split("/")[-1]
    pid = name_to_id.get(s)
    return int(pid) if pid is not None else (hash(s) % 10**9)

# ─────────────────────────────────────────────────────────────────────
# 2025-26 new format → internal standard
# ─────────────────────────────────────────────────────────────────────
def convert_new_pbp(pbp_df, name_to_id={}):
    """
    Convert the 2025-26 combined-stats CSV to internal standard columns.
    """
    df = pbp_df.copy()
    df["GAME_ID"]      = df["game_id"]
    df["PERIOD"]       = df["period"].astype(int)
    df["HOME_SCORE"]   = pd.to_numeric(df["home_score"],  errors="coerce").fillna(0)
    df["AWAY_SCORE"]   = pd.to_numeric(df["away_score"],  errors="coerce").fillna(0)
    df["game_date"]    = pd.to_datetime(df["date"], errors="coerce")

    # ── Event type mapping ──────────────────────────────────────────
    cond = [
        (df["event_type"] == "shot") & (df["result"] == "made"),
        (df["event_type"] == "shot") & (df["result"] == "missed"),
        (df["event_type"] == "free throw"),
        (df["event_type"] == "rebound"),
        (df["event_type"] == "turnover"),
        (df["event_type"] == "foul"),
        (df["event_type"] == "substitution"),
    ]
    df["EVENTMSGTYPE"] = np.select(cond, [1, 2, 3, 4, 5, 6, 8], default=0)
    df["EVENTMSGACTIONTYPE"] = np.where(
        (df["event_type"] == "free throw") & (df["type"] == "1 of 1"), 16, 0)

    df["points"] = pd.to_numeric(df["points"], errors="coerce").fillna(0).astype(int)

    # ── Descriptions ───────────────────────────────────────────────
    df["HOMEDESCRIPTION"]   = np.where(df["team"] == df["home_team"], df["description"], np.nan)
    df["VISITORDESCRIPTION"] = np.where(df["team"] == df["away_team"], df["description"], np.nan)

    # ── Player IDs ─────────────────────────────────────────────────
    df["PLAYER1_ID"] = df["player"].apply(lambda x: _map_player_name(x, name_to_id))
    df["PLAYER2_ID"] = df["assist"].apply(lambda x: _map_player_name(x, name_to_id))
    df["PLAYER3_ID"] = df["block"].apply(lambda x: _map_player_name(x, name_to_id))

    # ── Lineup strings ─────────────────────────────────────────────
    h_id_cols = [f"h{i}" for i in range(1, 6)]
    a_id_cols = [f"a{i}" for i in range(1, 6)]

    def _lineup(row, cols):
        ids = [_map_player_name(row[c], name_to_id) for c in cols if c in row.index]
        ids = sorted(int(x) for x in ids if x is not None)
        return "-".join(map(str, ids))

    df["HOME_players"] = df.apply(lambda r: _lineup(r, h_id_cols), axis=1)
    df["AWAY_players"] = df.apply(lambda r: _lineup(r, a_id_cols), axis=1)

    # Drop rows with no lineup info
    df = df[(df["HOME_players"] != "") & (df["AWAY_players"] != "")].reset_index(drop=True)
    return df

# ─────────────────────────────────────────────────────────────────────
# V3-format (22-23 / 23-24 / 24-25) → internal standard
# ─────────────────────────────────────────────────────────────────────
def convert_v3_pbp(pbp_df):
    """
    V3 fallback parsing. Handles completely missing game_date and team names
    by dynamically inferring them from gameId prefixes and location identifiers.
    """
    df = pbp_df.copy()
    col = {c.lower(): c for c in df.columns}

    def _get(std, *alts):
        for a in [std] + list(alts):
            if a in col: return df[col[a]]
        return pd.Series(np.nan, index=df.index)

    df["GAME_ID"]     = _get("gameid", "game_id")
    df["PERIOD"]      = pd.to_numeric(_get("period"), errors="coerce").fillna(1).astype(int)
    df["HOME_SCORE"]  = pd.to_numeric(_get("scorehome"),   errors="coerce").fillna(0)
    df["AWAY_SCORE"]  = pd.to_numeric(_get("scoreaway"),   errors="coerce").fillna(0)

    # ── Missing column fallback logic for Game Date & Teams ────────
    if "game_date" in col or "date" in col:
        df["game_date"] = pd.to_datetime(_get("game_date", "date"), errors="coerce")
    else:
        # Fallback: Extract year from GameID (e.g., 22201218 -> 2022) to satisfy sort
        df["game_date"] = df["GAME_ID"].astype(str).apply(
            lambda x: pd.to_datetime(f"20{x[1:3]}-01-01") if len(x) >= 8 else pd.NaT
        )

    if "home_team" in col and "away_team" in col:
        df["home_team"] = _get("home_team")
        df["away_team"] = _get("away_team")
    else:
        # Fallback: Infer from 'location' flag and 'teamtricode'
        if "location" in col and "teamtricode" in col:
            home_map = df[df[col["location"]] == "h"].groupby("GAME_ID")[col["teamtricode"]].first()
            away_map = df[df[col["location"]] == "v"].groupby("GAME_ID")[col["teamtricode"]].first()
            df["home_team"] = df["GAME_ID"].map(home_map)
            df["away_team"] = df["GAME_ID"].map(away_map)

    # ── Action type ────────────────────────────────────────────────
    at = _get("actiontype", "event_type").str.lower().fillna("")
    st = _get("subtype", "type").str.lower().fillna("")
    cond = [
        (at == "made shot") | ((at == "missed shot") & (st == "") & st.str.contains("made", na=False)),
        at.str.contains("missed shot|miss", na=False),
        at.str.contains("free throw", na=False),
        at.str.contains("rebound", na=False),
        at.str.contains("turnover", na=False),
        at.str.contains("foul", na=False),
        at.str.contains("substitution", na=False),
    ]
    df["EVENTMSGTYPE"] = np.select(cond, [1, 2, 3, 4, 5, 6, 8], default=0)
    df["EVENTMSGACTIONTYPE"] = 0

    desc = _get("description").fillna("")
    loc  = _get("location").fillna("")
    df["HOMEDESCRIPTION"]    = np.where(loc == "h", desc, np.nan)
    df["VISITORDESCRIPTION"] = np.where(loc == "v", desc, np.nan)

    df["PLAYER1_ID"] = pd.to_numeric(_get("personid", "player1_id"), errors="coerce")
    df["PLAYER2_ID"] = pd.to_numeric(_get("player2_id"), errors="coerce")
    df["PLAYER3_ID"] = pd.to_numeric(_get("player3_id"), errors="coerce")

    # ── Lineup strings from HOME_PLAYERS_ON / AWAY_PLAYERS_ON ─────
    if "HOME_PLAYERS_ON" in df.columns and "AWAY_PLAYERS_ON" in df.columns:
        df["HOME_players"] = df["HOME_PLAYERS_ON"].apply(
            lambda s: "-".join(sorted(str(x) for x in _parse_player_string(s))) if isinstance(s, str) else "")
        df["AWAY_players"] = df["AWAY_PLAYERS_ON"].apply(
            lambda s: "-".join(sorted(str(x) for x in _parse_player_string(s))) if isinstance(s, str) else "")
    else:
        df["HOME_players"] = ""
        df["AWAY_players"] = ""

    df["points"] = pd.to_numeric(_get("pointstotal", "points"), errors="coerce").fillna(0).astype(int)
    df = df[(df["HOME_players"] != "") & (df["AWAY_players"] != "")].reset_index(drop=True)
    return df

print("✅ PBP converters ready and patched for missing data.")

✅ PBP converters ready and patched for missing data.


## Cell 4 — Preprocessing: xPoints, Possession Counting, Garbage-Time Filtering

In [ ]:
import pandas as pd
import numpy as np
import re

def preprocess_pbp(df, compute_xpoints=True, zone_pps=None):
    """
    Preprocess play‑by‑play data for possession & expected points calculations.

    Parameters
    ----------
    df : pd.DataFrame
        Raw PBP data (must contain required columns).
    compute_xpoints : bool, default True
        If False, skip xPoints calculation entirely.
    zone_pps : dict or None, default None
        Pre‑computed expected points per shot for each zone.
        Expected keys: 'ab3', 'rim', 'short_mid', 'long_mid', 'other'.
        If None, fixed league‑average defaults are used (non‑leaky).

    Returns
    -------
    pd.DataFrame
        Enriched DataFrame with added columns.
    """
    # Avoid copying to save memory (modifies df in‑place)
    is_home = df["HOMEDESCRIPTION"].notna() & df["VISITORDESCRIPTION"].isna()
    is_away = df["VISITORDESCRIPTION"].notna() & df["HOMEDESCRIPTION"].isna()
    is_make = (df["EVENTMSGTYPE"] == 1)
    is_miss = (df["EVENTMSGTYPE"] == 2)
    is_tov  = (df["EVENTMSGTYPE"] == 5)
    is_foul = (df["EVENTMSGTYPE"] == 6)

    df["is_fg_make"] = is_make.astype(int)
    df["is_fg_miss"] = is_miss.astype(int)
    df["is_tov"]     = is_tov.astype(int)

    combo_desc = df["HOMEDESCRIPTION"].fillna("") + " " + df["VISITORDESCRIPTION"].fillna("")

    # ── Shot zones & xPoints ───────────────────────────────────────
    if compute_xpoints:
        # Extract shot distance from description if column missing
        if "shot_distance" not in df.columns or df["shot_distance"].isna().all():
            df["shot_distance"] = combo_desc.str.extract(r"(\d+)\s*(?:'|-foot| foot| ft)").astype(float)
            df["shot_distance"] = df["shot_distance"].fillna(-1)

        if "type" not in df.columns or df["type"].isna().all():
            df["type"] = combo_desc.str.lower()

        dist = pd.to_numeric(df.get("shot_distance", pd.Series([-1]*len(df))), errors="coerce").fillna(-1)
        combo = df.get("type", pd.Series([""] * len(df))).fillna("").str.lower()

        # Safety net for missing distances (text‑based heuristics)
        is_3pt_text = combo.str.contains("3pt|three", na=False)
        is_rim_text = combo.str.contains("layup|dunk|tip", na=False)
        is_floater_text = combo.str.contains("floater|hook", na=False)
        is_jumper_text = combo.str.contains("jump|fadeaway|bank", na=False)

        dist = np.where((dist == -1) & is_3pt_text, 25.0, dist)
        dist = np.where((dist == -1) & is_rim_text, 1.0, dist)
        dist = np.where((dist == -1) & is_floater_text, 7.0, dist)
        dist = np.where((dist == -1) & is_jumper_text & ~is_3pt_text, 15.0, dist)
        df["shot_distance"] = dist

        is_3pt  = combo.str.contains("3pt", na=False) | combo.str.contains("three", na=False)
        if is_3pt.sum() == 0:
            is_3pt = (dist >= 23.75)

        is_rim     = combo.str.contains("layup|dunk|tip", na=False) | ((dist >= 0) & (dist <= 3))
        is_shortmd = (dist > 3)  & (dist <= 10)
        is_longmd  = (dist > 10) & (dist < 23.75) & ~is_3pt

        zone = np.select([is_3pt, is_rim, is_shortmd, is_longmd],
                         ["ab3", "rim", "short_mid", "long_mid"], default="other")
        df["shot_zone"] = zone

        # ─── LEAKAGE FIX: use supplied zone_pps or fixed defaults ───
        if zone_pps is None:
            # Safe, non‑leaky league averages (e.g., from previous season)
            zone_pps = {
                "ab3": 1.06,
                "rim": 1.20,
                "short_mid": 0.85,
                "long_mid": 0.90,
                "other": 0.90
            }
        # Ensure 'other' exists
        zone_pps.setdefault("other", 0.90)

        is_shot = is_make | is_miss
        df["xPoints"] = df["shot_zone"].map(zone_pps)
        df.loc[~is_shot, "xPoints"] = 0.0
    else:
        df["xPoints"] = 0.0

    # ── Actual points added ────────────────────────────────────────
    df["home_pts_added"] = 0
    df["away_pts_added"] = 0

    df.loc[is_make & is_home, "home_pts_added"] = np.where(
        combo_desc[is_make & is_home].str.contains("3PT", flags=re.IGNORECASE, na=False), 3, 2)

    df.loc[is_make & is_away, "away_pts_added"] = np.where(
        combo_desc[is_make & is_away].str.contains("3PT", flags=re.IGNORECASE, na=False), 3, 2)

    ft_made = (df["EVENTMSGTYPE"] == 3) & ~combo_desc.str.contains("MISS", flags=re.IGNORECASE, na=False)

    df.loc[ft_made & is_home, "home_pts_added"] = 1
    df.loc[ft_made & is_away, "away_pts_added"] = 1

    # ── Expected points added ──────────────────────────────────────
    is_shot2 = is_make | is_miss
    df["home_xpts_added"] = 0.0
    df["away_xpts_added"] = 0.0
    df.loc[is_shot2 & is_home, "home_xpts_added"] = df.loc[is_shot2 & is_home, "xPoints"]
    df.loc[is_shot2 & is_away, "away_xpts_added"] = df.loc[is_shot2 & is_away, "xPoints"]
    ft_att = (df["EVENTMSGTYPE"] == 3)
    ft_pct = ft_made.sum() / ft_att.sum() if ft_att.sum() > 0 else 0.77
    df.loc[ft_att & is_home, "home_xpts_added"] = ft_pct
    df.loc[ft_att & is_away, "away_xpts_added"] = ft_pct

    # ── True possession endings ────────────────────────────────────
    is_final_ft = ft_att & combo_desc.str.contains(r"1 of 1|2 of 2|3 of 3", regex=True, na=False)
    is_tech    = ft_att & (df.get("EVENTMSGACTIONTYPE", pd.Series(0, index=df.index)) == 16)
    is_and1    = ft_att & (df["EVENTMSGTYPE"].shift(1) == 1) & (df["PLAYER1_ID"] == df["PLAYER1_ID"].shift(1))
    df["is_true_ft_trip"] = (is_final_ft & ~is_and1 & ~is_tech).astype(int)

    # ── Offensive rebounds ────────────────────────────────────────
    df["shot_team_state"] = np.nan
    df.loc[is_miss & is_home, "shot_team_state"] = "home"
    df.loc[is_miss & is_away, "shot_team_state"] = "away"
    reset_mask = df["EVENTMSGTYPE"].isin([1, 5, 8]) | (df["PERIOD"] != df["PERIOD"].shift(1))
    df.loc[reset_mask, "shot_team_state"] = "RESET"

    df["last_shot_team"] = df["shot_team_state"].ffill(inplace=False)
    df.loc[df["last_shot_team"] == "RESET", "last_shot_team"] = np.nan

    df["home_oreb"] = ((df["EVENTMSGTYPE"] == 4) & is_home & (df["last_shot_team"] == "home")).astype(int)
    df["away_oreb"] = ((df["EVENTMSGTYPE"] == 4) & is_away & (df["last_shot_team"] == "away")).astype(int)

    # ── Turnovers forced, fouls drawn, blocks ─────────────────────
    df["home_tovs_forced"] = (is_tov & is_away).astype(int)
    df["away_tovs_forced"] = (is_tov & is_home).astype(int)
    df["home_fouls_drawn"] = (is_foul & is_away).astype(int)
    df["away_fouls_drawn"] = (is_foul & is_home).astype(int)
    if "PLAYER3_ID" in df.columns:
        df["home_blks"] = (is_miss & is_away & df["PLAYER3_ID"].notna()).astype(int)
        df["away_blks"] = (is_miss & is_home & df["PLAYER3_ID"].notna()).astype(int)
    else:
        df["home_blks"] = df["away_blks"] = 0

    # ── Possessions (Kubatko formula) ─────────────────────────────
    df["home_poss"] = ((df["is_fg_make"] + df["is_fg_miss"] + df["is_tov"]) * is_home.astype(int)
                       - df["home_oreb"]
                       + df["is_true_ft_trip"] * is_home.astype(int))

    df["away_poss"] = ((df["is_fg_make"] + df["is_fg_miss"] + df["is_tov"]) * is_away.astype(int)
                       - df["away_oreb"]
                       + df["is_true_ft_trip"] * is_away.astype(int))

    df["total_possessions"] = (df["home_poss"] + df["away_poss"])

    # ── Garbage time ─────────────────────────────────────────────
    def _min_rem(t):
        if pd.isna(t): return 12.0
        parts = str(t).split(":")
        return int(parts[0]) + int(parts[1]) / 60.0 if len(parts) == 2 else 12.0

    if "remaining_time" in df.columns:
        df["min_rem"] = df["remaining_time"].apply(_min_rem)
    else:
        df["min_rem"] = 12.0

    df["abs_margin"] = (pd.to_numeric(df.get("HOME_SCORE", pd.Series([0]*len(df))), errors="coerce").fillna(0) -
                        pd.to_numeric(df.get("AWAY_SCORE", pd.Series([0]*len(df))), errors="coerce").fillna(0)).abs()

    df["garbage"] = (df["PERIOD"] == 4) & (
        (df["abs_margin"] >= 20) |
        ((df["abs_margin"] >= 15) & (df["min_rem"] <= 3.0))
    )

    gc = ["total_possessions", "home_pts_added", "away_pts_added",
          "home_xpts_added",   "away_xpts_added"]
    df.loc[df["garbage"], gc] = 0.0

    return df

print("✅ Leakage‑free preprocessing function ready.")

✅ Leakage‑free preprocessing function ready.


## Cell 5 — Stint Builder with Usage Tracking

In [ ]:
import pandas as pd

def build_stints(pbp_df, assist_split=0.65):
    # 1. Initialize DF IMMEDIATELY
    df = pbp_df.copy()

    # 2. Check if the dataframe is empty or missing necessary columns
    if df.empty or "total_possessions" not in df.columns:
        print("⚠️ Warning: Dataframe empty or missing columns. Returning empty stints.")
        return pd.DataFrame()

    # 3. Possession Safety Check
    if df["total_possessions"].sum() == 0:
        print("⚠️ Warning: Dataframe contains 0 total possessions. Check your Preprocessing.")
        return pd.DataFrame()

    # 4. Grouping Logic
    df["stint_id"] = (
        (df["HOME_players"] != df["HOME_players"].shift()) |
        (df["AWAY_players"] != df["AWAY_players"].shift()) |
        (df["PERIOD"]       != df["PERIOD"].shift())
    ).cumsum()

    stints = df.groupby("stint_id").agg(
        GAME_ID            = ("GAME_ID",           "first"),
        game_date          = ("game_date",         "first"),
        home_team          = ("home_team",         "first"),
        away_team          = ("away_team",         "first"),
        PERIOD             = ("PERIOD",            "first"),
        HOME_players       = ("HOME_players",      "first"),
        AWAY_players       = ("AWAY_players",      "first"),
        HOME_SCORE_START   = ("HOME_SCORE",        "first"),
        AWAY_SCORE_START   = ("AWAY_SCORE",        "first"),
        home_pts           = ("home_pts_added",    "sum"),
        away_pts           = ("away_pts_added",    "sum"),
        home_xpts          = ("home_xpts_added",   "sum"),
        away_xpts          = ("away_xpts_added",   "sum"),
        possessions        = ("total_possessions", "sum"),
        home_oreb          = ("home_oreb",         "sum"),
        away_oreb          = ("away_oreb",         "sum"),
        home_tovs_forced   = ("home_tovs_forced",  "sum"),
        away_tovs_forced   = ("away_tovs_forced",  "sum"),
        home_fouls_drawn   = ("home_fouls_drawn",  "sum"),
        away_fouls_drawn   = ("away_fouls_drawn",  "sum"),
    ).reset_index()

    stints["HOME_SCORE_END"] = stints["HOME_SCORE_START"] + stints["home_pts"]
    stints["AWAY_SCORE_END"] = stints["AWAY_SCORE_START"] + stints["away_pts"]

    # 5. Build Usage Dicts (Raw counts for speed)
    home_u = {row["stint_id"]: {p: [0.0, 0.0, 0.0] for p in str(row["HOME_players"]).split("-") if p and p != "nan"}
              for _, row in stints.iterrows()}
    away_u = {row["stint_id"]: {p: [0.0, 0.0, 0.0] for p in str(row["AWAY_players"]).split("-") if p and p != "nan"}
              for _, row in stints.iterrows()}

    records = df.to_dict("records")
    for i, row in enumerate(records):
        sid  = row["stint_id"]
        p1   = str(int(row["PLAYER1_ID"])) if pd.notna(row.get("PLAYER1_ID")) else None
        p2   = str(int(row["PLAYER2_ID"])) if pd.notna(row.get("PLAYER2_ID")) else None
        is_mk = row.get("is_fg_make", 0) == 1
        is_ms = row.get("is_fg_miss", 0) == 1
        is_tv = row.get("is_tov",     0) == 1
        is_ft = row.get("is_true_ft_trip", 0) == 1

        nxt_oreb = False
        if is_ms and i + 1 < len(records):
            nx = records[i + 1]
            if nx.get("EVENTMSGTYPE") == 4 and nx.get("PLAYER1_ID") == row.get("PLAYER1_ID"):
                nxt_oreb = True

        for usage in (home_u.get(sid, {}), away_u.get(sid, {})):
            if is_mk:
                if p2 in usage and p1 in usage:
                    usage[p1][1] += 1.0
                    usage[p2][2] += 1.0
                elif p1 in usage:
                    usage[p1][0] += 1.0
            elif is_ms and not nxt_oreb:
                if p1 in usage: usage[p1][0] += 1.0
            elif is_tv or is_ft:
                if p1 in usage: usage[p1][0] += 1.0

    stints["home_usage"] = stints["stint_id"].map(home_u)
    stints["away_usage"] = stints["stint_id"].map(away_u)

    return stints[stints["possessions"] > 0].reset_index(drop=True)

print("✅ Stint builder rebuilt safely.")

✅ Stint builder rebuilt safely.


## Cell 6 — Player Rating Tracker (xPPP Elo)

In [ ]:
def lineup_uncertainty(self, player_ids):
    ids = [str(x) for x in player_ids if x and str(x) != "nan"]
    if not ids:
        return 350.0, 350.0
    o_rd = np.mean([self._get(p)["O_rd"] for p in ids])
    d_rd = np.mean([self._get(p)["D_rd"] for p in ids])
    return o_rd, d_rd

In [ ]:
# Cell 6 – Enhanced Player Rating Tracker (Glicko‑2 style, with lineup uncertainty and roster-aware offseason reversion)

import math
from collections import defaultdict
import numpy as np

class PlayerRatingTracker:
    """
    Glicko‑2 inspired rating tracker with separate offensive/defensive ratings.
    Maintains μ (rating), RD (uncertainty), and simplified volatility tracking.
    Exposes O_Elo / D_Elo for compatibility with existing feature code.
    """

    DEFAULTS = {
        "tau": 0.5,
        "epsilon": 1e-6,
        "default_mu": 1500.0,
        "default_rd": 350.0,
        "default_sigma": 0.06,
        "HOME_PPP_BOOST": 0.024,
        "OFFSEASON_REVERSION": 0.25,
        "USAGE_FLOOR": 0.15,
        "assist_split": 0.65,
        "ELO_SCALING_FACTOR": 1000,
    }

    def __init__(self, config=None, league_xppp=1.10):
        self.cfg = {**self.DEFAULTS, **(config or {})}
        self.league_xppp = league_xppp
        self.players = {}          # player_id -> rating dict
        self.player_games = defaultdict(int)

    def _get(self, pid):
        pid = str(pid)
        if pid not in self.players:
            self.players[pid] = {
                "O_mu": self.cfg["default_mu"],
                "D_mu": self.cfg["default_mu"],
                "O_rd": self.cfg["default_rd"],
                "D_rd": self.cfg["default_rd"],
                "O_sigma": self.cfg["default_sigma"],
                "D_sigma": self.cfg["default_sigma"],
                "Possessions": 0,
                "last_date": None,
            }
        return self.players[pid]

    @property
    def O_Elo(self):
        """For compatibility: returns offensive ratings dict."""
        return {pid: data["O_mu"] for pid, data in self.players.items()}

    @property
    def D_Elo(self):
        """For compatibility: returns defensive ratings dict."""
        return {pid: data["D_mu"] for pid, data in self.players.items()}

    # ------------------------------------------------------------------
    # Uncertainty (RD) methods – used by feature generator
    # ------------------------------------------------------------------
    def lineup_uncertainty(self, player_ids):
        """
        Returns mean offensive RD and defensive RD across the lineup.
        High RD means high uncertainty (few games, long inactivity, or new player).
        """
        ids = [str(x) for x in player_ids if x and str(x) != "nan"]
        if not ids:
            return 350.0, 350.0
        o_rd = np.mean([self._get(p)["O_rd"] for p in ids])
        d_rd = np.mean([self._get(p)["D_rd"] for p in ids])
        return o_rd, d_rd

    # ------------------------------------------------------------------
    # Inactivity decay (inflate RD when player hasn't played recently)
    # ------------------------------------------------------------------
    def apply_inactivity_decay(self, player_ids, date):
        for pid in player_ids:
            p = self._get(pid)
            if p["last_date"] is not None:
                days = (date - p["last_date"]).days
                if days > 60:
                    # Inflate RD linearly with weeks inactive (capped at 350)
                    p["O_rd"] = min(350.0, p["O_rd"] * (1 + 0.1 * (days / 7)))
                    p["D_rd"] = min(350.0, p["D_rd"] * (1 + 0.1 * (days / 7)))
            p["last_date"] = date

    # ------------------------------------------------------------------
    # Offseason reversion with roster continuity awareness
    # ------------------------------------------------------------------
    def offseason_revert(self, returning_player_ids=None):
        """
        Apply rating reversion and uncertainty reset at season boundary.
        If returning_player_ids is a set of player IDs expected to return,
        players NOT in that set revert more heavily (higher reversion factor).
        """
        base_rev = self.cfg["OFFSEASON_REVERSION"]
        default_rd = self.cfg["default_rd"]
        default_sigma = self.cfg["default_sigma"]

        for pid, p in self.players.items():
            # Determine effective reversion strength
            if returning_player_ids is not None and pid not in returning_player_ids:
                # Departing player: revert twice as much toward 1500
                effective_rev = min(base_rev * 2.0, 0.50)
            else:
                effective_rev = base_rev

            # Regress μ toward 1500
            p["O_mu"] = 1500.0 + (p["O_mu"] - 1500.0) * (1 - effective_rev)
            p["D_mu"] = 1500.0 + (p["D_mu"] - 1500.0) * (1 - effective_rev)

            # Reset RD and sigma to defaults (full uncertainty)
            p["O_rd"] = default_rd
            p["D_rd"] = default_rd
            p["O_sigma"] = default_sigma
            p["D_sigma"] = default_sigma
            p["Possessions"] = 0

        self.player_games.clear()

    # ------------------------------------------------------------------
    # Lineup aggregate stats (mean mu and experience)
    # ------------------------------------------------------------------
    def lineup_stats(self, player_ids):
        ids = [str(x) for x in player_ids if x and str(x) != "nan"]
        if not ids:
            return 1500.0, 1500.0, 0.0
        off = np.mean([self._get(p)["O_mu"] for p in ids])
        dff = np.mean([self._get(p)["D_mu"] for p in ids])
        exp = np.mean([self._get(p)["Possessions"] for p in ids])
        return off, dff, exp

    # ------------------------------------------------------------------
    # Core update: process a lineup stint (one or more possessions)
    # ------------------------------------------------------------------
    def process_stint(self, ids_A, ids_B, poss, xpts_A, xpts_B,
                      usage_A=None, usage_B=None,
                      period=1, start_A=0, start_B=0,
                      end_A=0, end_B=0, season_progress=0.5):
        ast_split = self.cfg.get("assist_split", 0.65)

        def collapse_usage(u_dict):
            if not u_dict:
                return {}
            return {p: (v[0] + v[1] * ast_split + v[2] * (1 - ast_split))
                    for p, v in u_dict.items()}

        flat_usage_A = collapse_usage(usage_A)
        flat_usage_B = collapse_usage(usage_B)

        off_A, def_A, _ = self.lineup_stats(ids_A)
        off_B, def_B, _ = self.lineup_stats(ids_B)
        margin_A = end_A - start_B
        wt = self._weight_stint(poss, period, margin_A, season_progress)

        scaling = self.cfg["ELO_SCALING_FACTOR"]
        h_boost = self.cfg["HOME_PPP_BOOST"]

        exp_ppp_A = self.league_xppp + h_boost + (off_A - def_B) / scaling
        exp_ppp_B = self.league_xppp - h_boost + (off_B - def_A) / scaling

        act_ppp_A = xpts_A / poss if poss > 0 else 0
        act_ppp_B = xpts_B / poss if poss > 0 else 0

        err_A = act_ppp_A - exp_ppp_A
        err_B = act_ppp_B - exp_ppp_B

        # Update offense of A and defense of B based on their own errors
        self._update_ratings(ids_A, err_A * wt, poss, flat_usage_A, side="off")
        self._update_ratings(ids_B, err_B * wt, poss, flat_usage_B, side="def")
        # Cross updates: defense of A affected by B's error, offense of B affected by A's error
        self._update_ratings(ids_A, -err_B * wt, poss, flat_usage_A, side="def")
        self._update_ratings(ids_B, -err_A * wt, poss, flat_usage_B, side="off")

    def _update_ratings(self, ids, error, poss, usage, side):
        """
        Update ratings for a single side (offense or defense).
        error = actual_xPPP - expected_xPPP  (positive means outperformed)
        usage: dict player -> usage weight (0..1)
        """
        ids = [str(x) for x in ids if x and str(x) != "nan"]
        if not ids:
            return

        n = len(ids)
        base_w = 1.0 / n
        total_usage = sum(usage.values()) if usage else 0
        u_floor = self.cfg["USAGE_FLOOR"]

        shares = []
        for p in ids:
            if total_usage > 0:
                raw = usage.get(p, 0) / total_usage
            else:
                raw = base_w
            shares.append(max(raw, base_w * u_floor))
        s = sum(shares)

        for i, p in enumerate(ids):
            pl = self._get(p)
            games = self.player_games[p]
            # Dynamic learning rate: higher for rookies, lower for veterans
            k_mult = max(0.5, 2.0 * math.exp(-games / 15))
            delta = error * 0.9 * (shares[i] / s) * k_mult

            if side == "off":
                pl["O_mu"] += delta
                # RD decays slightly after each update (but never below 30)
                pl["O_rd"] = max(30.0, min(350.0, pl["O_rd"] * 0.99 + 2.0))
            else:  # defense
                pl["D_mu"] += delta
                pl["D_rd"] = max(30.0, min(350.0, pl["D_rd"] * 0.99 + 2.0))

            pl["Possessions"] += poss
            self.player_games[p] += 1

    @staticmethod
    def _weight_stint(poss, period, margin, season_progress):
        """Weight a stint by possessions, downweighting garbage time."""
        weight = poss
        if period >= 4 and abs(margin) >= 15:
            weight *= 0.5
        return weight


print("✅ Enhanced PlayerRatingTracker ready (Glicko‑2 style, with lineup uncertainty + roster‑aware reversion).")

✅ Enhanced PlayerRatingTracker ready (Glicko‑2 style, with lineup uncertainty + roster‑aware reversion).


## Cell 7 — Hierarchical Lineup Possession Engine

In [ ]:
class HierarchicalPossessionEngine:
    def __init__(self, w1=0.50, w2=0.25, w3=0.15, w5=0.10,
                 k_off=2.0, k_def=1.5, league_avg_rtg=DEFAULT_LEAGUE_RTG):
        total = w1 + w2 + w3 + w5
        self.W = {1: w1/total, 2: w2/total, 3: w3/total, 5: w5/total}

        self.K_off = {1: k_off, 2: k_off*0.50, 3: k_off*0.25, 5: k_off*0.10}
        self.K_def = {1: k_def, 2: k_def*0.50, 3: k_def*0.25, 5: k_def*0.10}

        self.lg = league_avg_rtg
        self.off = defaultdict(float)
        self.dff = defaultdict(float)

    def _combos(self, lineup):
        ids = sorted(int(x) for x in lineup if x is not None and str(x) != "nan")
        return {
            1: [(p,) for p in ids],
            2: list(itertools.combinations(ids, 2)),
            3: list(itertools.combinations(ids, 3)),
            5: [tuple(ids)] if len(ids) == 5 else [],
        }

    def _mean(self, combos, store):
        return float(np.mean([store[c] for c in combos])) if combos else 0.0

    def lineup_rating(self, lineup):
        cb = self._combos(lineup)
        if not cb[1]: return 0.0, 0.0
        off = sum(self.W[l] * self._mean(cb[l], self.off) for l in self.W)
        dff = sum(self.W[l] * self._mean(cb[l], self.dff) for l in self.W)
        return off, dff

    def predict_pts(self, off_ln, def_ln, possessions):
        cb_off = self._combos(off_ln); cb_def = self._combos(def_ln)
        os = sum(self.W[l] * self._mean(cb_off[l], self.off) for l in self.W)
        ds = sum(self.W[l] * self._mean(cb_def[l], self.dff) for l in self.W)
        o2 = sum(self.W[l] * self._mean(cb_def[l], self.off) for l in self.W)
        d2 = sum(self.W[l] * self._mean(cb_off[l], self.dff) for l in self.W)
        p = possessions / 100.0
        return (self.lg + os - ds) * p, (self.lg + o2 - d2) * p, cb_off, cb_def

    def update(self, off_ln, def_ln, pts_off, pts_def, possessions):
        if possessions <= 0: return
        xo, xd, cb_off, cb_def = self.predict_pts(off_ln, def_ln, possessions)

        eo, ed = pts_off - xo, pts_def - xd

        for l in [1, 2, 3, 5]:
            ko, kd = self.K_off[l], self.K_def[l]
            for c in cb_off[l]:
                self.off[c] += ko * eo
                self.dff[c] -= kd * ed
            for c in cb_def[l]:
                self.off[c] += ko * ed
                self.dff[c] -= kd * eo

    def offseason_revert(self):
        for store in (self.off, self.dff):
            for k in list(store): store[k] *= (1.0 - OFFSEASON_REVERSION)

print("✅ HierarchicalPossessionEngine ready.")

✅ HierarchicalPossessionEngine ready.


## Cell 8 — Odds / Market Helpers

In [ ]:
# Cell 8 – Odds / Market Helpers (updated)

import pandas as pd
import numpy as np
import datetime
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────────────────────────────────────
# 1. LEAKAGE‑FREE ODDS RETRIEVAL (ONLY PAST DATES)
# ──────────────────────────────────────────────────────────────────────────────

def get_odds(game_date, home_team, odds_dict):
    """
    Directly fetches the odds for the specific game date.
    """
    if not odds_dict:
        return np.nan, np.nan

    try:
        game_dt = pd.to_datetime(game_date).date()
    except:
        return np.nan, np.nan

    # Team aliases (same as before, but kept minimal for brevity)
    ABBR_MAP = {
        'Atlanta': 'ATL', 'Boston': 'BOS', 'Brooklyn': 'BKN', 'Charlotte': 'CHA',
        'Chicago': 'CHI', 'Cleveland': 'CLE', 'Dallas': 'DAL', 'Denver': 'DEN',
        'Detroit': 'DET', 'Golden State': 'GSW', 'Houston': 'HOU', 'Indiana': 'IND',
        'LA Clippers': 'LAC', 'Los Angeles Clippers': 'LAC', 'Clippers': 'LAC',
        'LA Lakers': 'LAL', 'Los Angeles Lakers': 'LAL', 'Lakers': 'LAL',
        'Memphis': 'MEM', 'Miami': 'MIA', 'Milwaukee': 'MIL', 'Minnesota': 'MIN',
        'New Orleans': 'NOP', 'New York': 'NYK', 'Knicks': 'NYK',
        'Oklahoma City': 'OKC', 'Orlando': 'ORL', 'Philadelphia': 'PHI', '76ers': 'PHI',
        'Phoenix': 'PHX', 'Portland': 'POR', 'Sacramento': 'SAC',
        'San Antonio': 'SAS', 'Toronto': 'TOR', 'Utah': 'UTA', 'Washington': 'WAS'
    }
    TEAM_ALIASES = {
        'ATL': ['ATL', 'Atlanta', 'Atlanta Hawks'], 'BOS': ['BOS', 'Boston', 'Boston Celtics'],
        'BKN': ['BKN', 'Brooklyn', 'Brooklyn Nets'], 'CHA': ['CHA', 'Charlotte', 'Charlotte Hornets'],
        'CHI': ['CHI', 'Chicago', 'Chicago Bulls'], 'CLE': ['CLE', 'Cleveland', 'Cleveland Cavaliers'],
        'DAL': ['DAL', 'Dallas', 'Dallas Mavericks'], 'DEN': ['DEN', 'Denver', 'Denver Nuggets'],
        'DET': ['DET', 'Detroit', 'Detroit Pistons'], 'GSW': ['GSW', 'Golden State', 'Golden State Warriors'],
        'HOU': ['HOU', 'Houston', 'Houston Rockets'], 'IND': ['IND', 'Indiana', 'Indiana Pacers'],
        'LAC': ['LAC', 'LA Clippers', 'Los Angeles Clippers', 'Clippers'],
        'LAL': ['LAL', 'LA Lakers', 'Los Angeles Lakers', 'Lakers'],
        'MEM': ['MEM', 'Memphis', 'Memphis Grizzlies'], 'MIA': ['MIA', 'Miami', 'Miami Heat'],
        'MIL': ['MIL', 'Milwaukee', 'Milwaukee Bucks'], 'MIN': ['MIN', 'Minnesota', 'Minnesota Timberwolves'],
        'NOP': ['NOP', 'New Orleans', 'New Orleans Pelicans'],
        'NYK': ['NYK', 'New York', 'New York Knicks', 'Knicks'],
        'OKC': ['OKC', 'Oklahoma City', 'Oklahoma City Thunder'], 'ORL': ['ORL', 'Orlando', 'Orlando Magic'],
        'PHI': ['PHI', 'Philadelphia', 'Philadelphia 76ers', '76ers'],
        'PHX': ['PHX', 'Phoenix', 'Phoenix Suns'], 'POR': ['POR', 'Portland', 'Portland Trail Blazers'],
        'SAC': ['SAC', 'Sacramento', 'Sacramento Kings'], 'SAS': ['SAS', 'San Antonio', 'San Antonio Spurs'],
        'TOR': ['TOR', 'Toronto', 'Toronto Raptors'], 'UTA': ['UTA', 'Utah', 'Utah Jazz'],
        'WAS': ['WAS', 'Washington', 'Washington Wizards']
    }

    abbr = ABBR_MAP.get(str(home_team).strip(), str(home_team).strip())
    aliases = TEAM_ALIASES.get(abbr, [home_team])

    # Direct lookup: (date, team_alias) -> (spread, ml)
    for alias in aliases:
        if (game_dt, alias) in odds_dict:
            vals = odds_dict[(game_dt, alias)]
            return vals.get('spread', np.nan), vals.get('ml', np.nan)

    return np.nan, np.nan

# We'll also need a function to get closing spread for CLV tracking.
# We'll implement a separate function to fetch closing line (latest available).
def get_closing_odds(game_date, home_team, odds_dict):
    """Return closing spread and moneyline (latest available before game)."""
    if not odds_dict:
        return np.nan, np.nan

    try:
        game_dt = pd.to_datetime(game_date).date()
        if pd.isnull(game_dt):
            return np.nan, np.nan
    except:
        return np.nan, np.nan

    # Same team mapping as above (reuse)
    ABBR_MAP = {...}  # include full mapping here, or define globally
    TEAM_ALIASES = {...}
    abbr = ABBR_MAP.get(str(home_team).strip(), str(home_team).strip())
    aliases = TEAM_ALIASES.get(abbr, [home_team])
    game_date = pd.to_datetime(game_date).date()
    valid_odds = { (d, t): v for (d, t), v in odds_dict.items() if d <= game_date }

    latest_date = None
    best_spread = np.nan
    best_ml = np.nan

    for (d, team), vals in valid_odds.items():
        if team in aliases:
            if latest_date is None or d > latest_date:
                latest_date = d
                best_spread = vals.get('spread', np.nan)
                best_ml = vals.get('ml', np.nan)

    return best_spread, best_ml

# ──────────────────────────────────────────────────────────────────────────────
# 2. EDGE AND EXPECTED VALUE CALCULATIONS (unchanged)
# ──────────────────────────────────────────────────────────────────────────────

def american_to_decimal(odds):
    if pd.isna(odds):
        return np.nan
    if odds > 0:
        return 1 + odds / 100.0
    else:
        return 1 + 100.0 / abs(odds)

def implied_probability(odds):
    if pd.isna(odds):
        return np.nan
    if odds > 0:
        return 100.0 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)

def spread_edge(model_spread, market_spread):
    if pd.isna(market_spread):
        return np.nan
    return model_spread + market_spread

def moneyline_edge(model_win_prob_home, market_ml_home):
    if pd.isna(market_ml_home):
        return np.nan
    dec = american_to_decimal(market_ml_home)
    ev = (model_win_prob_home * dec) - 1.0
    return ev

def bet_analysis(model_spread, market_spread, model_win_prob=None, market_ml=None,
                 min_edge_pts=2.5, min_ev=0.03, conf_width=None):
    result = {
        "spread_edge_pts": np.nan,
        "spread_direction": "Pass",
        "spread_confidence": 0,
        "spread_stars": "No market",
        "ml_ev": np.nan,
        "ml_direction": "Pass",
        "ml_confidence": 0
    }

    if not pd.isna(market_spread):
        edge_pts = spread_edge(model_spread, market_spread)
        result["spread_edge_pts"] = edge_pts
        abs_edge = abs(edge_pts)
        if abs_edge >= min_edge_pts:
            direction = "Home" if edge_pts > 0 else "Away"
            result["spread_direction"] = direction
            conf = min(100, int((abs_edge / 12.0) * 50))
            result["spread_confidence"] = conf
            if abs_edge >= 8:
                result["spread_stars"] = "⭐⭐⭐⭐ (Strong)"
            elif abs_edge >= 5:
                result["spread_stars"] = "⭐⭐⭐ (Good)"
            else:
                result["spread_stars"] = "⭐⭐ (Lean)"
        else:
            result["spread_direction"] = "Pass"
            result["spread_stars"] = "Pass"

    if (model_win_prob is not None) and not pd.isna(market_ml):
        ev_home = moneyline_edge(model_win_prob, market_ml)
        ev_away = moneyline_edge(1 - model_win_prob, -market_ml)
        result["ml_ev"] = ev_home
        if ev_home > min_ev:
            result["ml_direction"] = "Home"
            result["ml_confidence"] = min(100, int(ev_home * 1000))
        elif ev_away > min_ev:
            result["ml_direction"] = "Away"
            result["ml_confidence"] = min(100, int(ev_away * 1000))
        else:
            result["ml_direction"] = "Pass"

    return result

# ──────────────────────────────────────────────────────────────────────────────
# 3. FINANCIAL CALCULATOR (unchanged)
# ──────────────────────────────────────────────────────────────────────────────

def calculate_financials(df, unit_size=10.0, use_fractional_kelly=False):
    df = df.copy()
    df["SPREAD_PROFIT"] = 0.0
    win_payout = unit_size * (100 / 110)

    if use_fractional_kelly and "KELLY_FRACTION" in df.columns:
        effective_unit = unit_size * df["KELLY_FRACTION"].fillna(0)
    else:
        effective_unit = unit_size

    mask_win = df["ATS_WIN"] == 1
    mask_loss = df["ATS_LOSS"] == 1
    df.loc[mask_win, "SPREAD_PROFIT"] = win_payout * (effective_unit / unit_size)
    df.loc[mask_loss, "SPREAD_PROFIT"] = -effective_unit

    df["ML_PROFIT"] = 0.0
    df["ML_BET"] = "Pass"
    if "MARKET_ML" in df.columns and "WIN_PROB" in df.columns:
        for idx, row in df.iterrows():
            ml = row["MARKET_ML"]
            if pd.isna(ml):
                continue
            model_prob_home = row["WIN_PROB"]
            dec_home = american_to_decimal(ml)
            dec_away = american_to_decimal(-ml)
            ev_home = (model_prob_home * dec_home) - 1
            ev_away = ((1 - model_prob_home) * dec_away) - 1

            if ev_home > 0.03:
                df.at[idx, "ML_BET"] = "Home"
                if row["ACTUAL_HOME"] > row["ACTUAL_AWAY"]:
                    profit = unit_size * (dec_home - 1)
                    df.at[idx, "ML_PROFIT"] = profit
                else:
                    df.at[idx, "ML_PROFIT"] = -unit_size
            elif ev_away > 0.03:
                df.at[idx, "ML_BET"] = "Away"
                if row["ACTUAL_AWAY"] > row["ACTUAL_HOME"]:
                    profit = unit_size * (dec_away - 1)
                    df.at[idx, "ML_PROFIT"] = profit
                else:
                    df.at[idx, "ML_PROFIT"] = -unit_size
    return df

# ──────────────────────────────────────────────────────────────────────────────
# 4. REPORTING AND GRAPHING (unchanged)
# ──────────────────────────────────────────────────────────────────────────────

def print_interval_ats_report(results_df):
    if results_df.empty:
        return None
    df = results_df[results_df["MARKET_SPREAD"].notna() & (results_df["DIRECTION"] != "Pass")].copy()
    if df.empty:
        print("  [No Vegas Market lines found or zero active bets triggered]")
        return None

    home_covers = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) > 0
    away_covers = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) < 0
    pushes = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) == 0

    df["ATS_WIN"] = (((df["DIRECTION"] == "Home") & home_covers) |
                     ((df["DIRECTION"] == "Away") & away_covers)).astype(int)
    df["ATS_LOSS"] = (((df["DIRECTION"] == "Home") & away_covers) |
                      ((df["DIRECTION"] == "Away") & home_covers)).astype(int)
    df["ATS_PUSH"] = pushes.astype(int)

    df = calculate_financials(df, unit_size=10.0)

    bins = [-1.0, 2.999, 4.999, 7.999, float('inf')]
    labels = ["0.0 - 2.9", "3.0 - 4.9", "5.0 - 7.9", "8.0+"]
    df["EDGE_TIER"] = pd.cut(df["EDGE"], bins=bins, labels=labels, right=True)

    print("══════════════════════════════════════════════════════════════")
    print("      SPREAD BETTING PERFORMANCE ($10 Base Unit)              ")
    print("══════════════════════════════════════════════════════════════")
    for tier in labels:
        tier_df = df[df["EDGE_TIER"] == tier]
        total = len(tier_df)
        if total == 0:
            continue
        w = tier_df["ATS_WIN"].sum()
        l = tier_df["ATS_LOSS"].sum()
        p = tier_df["ATS_PUSH"].sum()
        win_pct = w / (w + l) if (w + l) > 0 else 0.0
        profit = tier_df["SPREAD_PROFIT"].sum()
        print(f" Edge {tier:>9} pts | {total:3d} bets | {w:2d}-{l:2d}-{p:1d} | {win_pct:5.1%} ATS | Net: ${profit:+.2f}")

    ml_df = df[df["ML_BET"] != "Pass"]
    if not ml_df.empty:
        ml_wins = (ml_df["ML_PROFIT"] > 0).sum()
        ml_losses = (ml_df["ML_PROFIT"] < 0).sum()
        ml_profit = ml_df["ML_PROFIT"].sum()
        ml_win_pct = ml_wins / len(ml_df)
        print("──────────────────────────────────────────────────────────────")
        print("      MONEYLINE VALUE PERFORMANCE ($10 Base Unit)             ")
        print("──────────────────────────────────────────────────────────────")
        print(f" Total ML Plays: {len(ml_df):3d} | {ml_wins:2d} Wins - {ml_losses:2d} Losses | {ml_win_pct:5.1%} Hit Rate")
        print(f" Total Net Profit: ${ml_profit:+.2f}")
    print("══════════════════════════════════════════════════════════════\n")
    return df

def plot_financial_performance(financial_df, title="Cumulative_Profit"):
    if financial_df is None or financial_df.empty:
        return
    df = financial_df.sort_values("DATE").reset_index(drop=True)
    df["CUM_SPREAD"] = df["SPREAD_PROFIT"].cumsum()
    plt.figure(figsize=(14, 7))
    plt.plot(df.index, df["CUM_SPREAD"], label="ATS Spread Profit ($10/bet)", color="#2ca02c", linewidth=2.5)
    if "ML_PROFIT" in df.columns:
        df["CUM_ML"] = df["ML_PROFIT"].cumsum()
        plt.plot(df.index, df["CUM_ML"], label="Moneyline Profit ($10/bet)", color="#1f77b4", linewidth=2.5, alpha=0.8)
    plt.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.6)
    plt.title(f"Simulation Equity Curve: {title}", fontsize=14, fontweight="bold")
    plt.xlabel("Number of Bets Placed (Chronological)", fontsize=12)
    plt.ylabel("Cumulative Profit ($)", fontsize=12)
    plt.legend(loc="upper left", fontsize=11)
    plt.grid(alpha=0.3)
    filename = f"equity_curve_{title.replace(' ', '_').lower()}.png"
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"✅ Equity curve graph saved as '{filename}'")

class SpreadCalibrator:
    def __init__(self, window=50):
        self.errors = deque(maxlen=window)
        self.bias = 0.0
    def update(self, pred, actual):
        self.errors.append(pred - actual)
        self.bias = np.mean(self.errors)
    def correct(self, pred):
        return pred - self.bias

print("✅ Leakage‑free Vegas helpers, ML/spread edge calculators, and financials loaded.")

✅ Leakage‑free Vegas helpers, ML/spread edge calculators, and financials loaded.


In [ ]:
# Cell 8b – Team xPPP Tracker (rolling 40 games, previous season weight 0.5)
from collections import defaultdict, deque

class TeamXpppTracker:
    """
    Maintains rolling offensive and defensive xPPP (expected points per possession)
    for each team, using a weighted window:
      - last 40 games: weight 1.0
      - previous season games: weight 0.5
    """
    def __init__(self, window_size=40, prev_season_weight=0.5):
        self.window_size = window_size
        self.prev_season_weight = prev_season_weight
        # For each team, store deque of (game_date, season, off_xppp, def_xppp)
        self.history = defaultdict(lambda: deque(maxlen=window_size))

    def update(self, team, game_date, season, off_xppp, def_xppp):
        """
        Add a game's team-level offensive and defensive xPPP.
        off_xppp = total_offensive_xPoints / total_offensive_possessions
        def_xppp = total_defensive_xPoints_allowed / total_defensive_possessions
        """
        self.history[team].append((game_date, season, off_xppp, def_xppp))

    def get_rolling_xppp(self, team, current_season, current_date):
        """
        Returns (rolling_off_xppp, rolling_def_xppp) using weighted average.
        Games in current season: weight 1.0.
        Games in previous season: weight prev_season_weight.
        Only games before current_date are considered.
        """
        if team not in self.history:
            return DEFAULT_LEAGUE_XPPP, DEFAULT_LEAGUE_XPPP

        total_weight = 0.0
        sum_off = 0.0
        sum_def = 0.0

        for (game_date, season, off_xppp, def_xppp) in self.history[team]:
            if game_date >= current_date:
                continue   # never use future data
            if season == current_season:
                w = 1.0
            else:
                w = self.prev_season_weight   # previous season
            total_weight += w
            sum_off += off_xppp * w
            sum_def += def_xppp * w

        if total_weight == 0:
            return DEFAULT_LEAGUE_XPPP, DEFAULT_LEAGUE_XPPP

        return sum_off / total_weight, sum_def / total_weight

## Cell 9 — Walk-Forward Feature Generator

In [ ]:
import requests

def get_live_inactives(team_abbr):
    """Return list of player IDs who are OUT for today's game (ESPN)."""
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    try:
        data = requests.get(url).json()
        out_ids = []
        for team in data.get('teams', []):
            if team['team']['abbreviation'] == team_abbr.upper():
                for injury in team.get('injuries', []):
                    if injury['status'] == 'Out':
                        player_name = injury['athlete']['displayName']
                        pid = name_to_id.get(player_name)
                        if pid:
                            out_ids.append(pid)
        return out_ids
    except Exception as e:
        print(f"⚠️ ESPN injury fetch failed: {e}")
        return []

In [ ]:
# Cell 9 – generate_features (rewritten with SOS and aligned with run_simulation)
from collections import defaultdict, deque
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

def generate_features(
    stints_df: pd.DataFrame,
    hier_engine,
    elo_tracker,
    pace_tracker,
    odds_dict: dict = None,
    update_engines: bool = True,
    team_xppp_tracker=None,
    sos_window: int = 15,          # rolling window for SOS
) -> pd.DataFrame:
    """
    Generate a feature DataFrame for all games in stints_df, ensuring NO data leakage.

    Added features:
      - h_sos, a_sos, sos_diff: rolling average net rating of opponents faced.
    """
    df = stints_df.copy()
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df = df.sort_values(["game_date", "GAME_ID", "stint_id"]).reset_index(drop=True)

    rows = []
    last_game_date = {}
    season_year = None
    season_start_date = None
    team_games_played = defaultdict(int)
    team_rosters_seen = defaultdict(set)

    # Recent form deques (net rating)
    RECENT_WINDOW = 10
    team_recent_net = defaultdict(lambda: deque(maxlen=RECENT_WINDOW))

    # --- SOS tracking ---
    opponent_history = defaultdict(lambda: deque(maxlen=sos_window))

    lineup_cache = {}
    total_games = len(df["GAME_ID"].unique()) if update_engines else 1

    for game_id, group in tqdm(df.groupby("GAME_ID", sort=False), desc="Building features"):
        gdate = group["game_date"].iloc[0]
        home_raw = group["home_team"].iloc[0]
        away_raw = group["away_team"].iloc[0]
        home = TEAM_MAP.get(home_raw, home_raw)
        away = TEAM_MAP.get(away_raw, away_raw)

        if isinstance(gdate, pd.Timestamp):
            current_season = gdate.year + (1 if gdate.month >= 9 else 0)
        else:
            current_season = 2025

        if update_engines:
            if season_year is None:
                season_year = gdate.year
            elif gdate.month > 8 and gdate.year > season_year:
                hier_engine.offseason_revert()
                elo_tracker.offseason_revert()
                season_year = gdate.year
                season_start_date = gdate
                team_games_played.clear()
                team_rosters_seen.clear()
                lineup_cache.clear()
                team_recent_net.clear()
                opponent_history.clear()   # reset SOS at season boundary

            if season_start_date is None:
                season_start_date = gdate

        # Starting lineups
        if home in lineup_cache:
            home_starters = lineup_cache[home]
        else:
            first_stint = group.iloc[0]
            home_starters = _parse_player_string(first_stint.get("HOME_players", ""))
            lineup_cache[home] = home_starters

        if away in lineup_cache:
            away_starters = lineup_cache[away]
        else:
            first_stint = group.iloc[0]
            away_starters = _parse_player_string(first_stint.get("AWAY_players", ""))
            lineup_cache[away] = away_starters

        # Aggregate actual outcomes
        act_h = act_a = tot_poss = 0.0
        home_tot_poss = away_tot_poss = 0.0
        home_tot_xpts = away_tot_xpts = 0.0

        for _, row in group.iterrows():
            act_h += float(row.get("home_pts", 0))
            act_a += float(row.get("away_pts", 0))
            tot_poss += float(row.get("possessions", 1.0))
            home_tot_poss += float(row.get("home_poss", 0))
            away_tot_poss += float(row.get("away_poss", 0))
            home_tot_xpts += float(row.get("home_xpts", 0))
            away_tot_xpts += float(row.get("away_xpts", 0))

        if home_tot_poss == 0:
            home_tot_poss = tot_poss / 2
        if away_tot_poss == 0:
            away_tot_poss = tot_poss / 2

        home_xppp_game = home_tot_xpts / home_tot_poss if home_tot_poss > 0 else DEFAULT_LEAGUE_XPPP
        away_xppp_game = away_tot_xpts / away_tot_poss if away_tot_poss > 0 else DEFAULT_LEAGUE_XPPP

        # Inactivity decay
        if update_engines and isinstance(gdate, pd.Timestamp):
            elo_tracker.apply_inactivity_decay(home_starters, gdate.date())
            elo_tracker.apply_inactivity_decay(away_starters, gdate.date())

        # Pre‑game ratings
        ho_off, ho_def, ho_exp = elo_tracker.lineup_stats(home_starters)
        ao_off, ao_def, ao_exp = elo_tracker.lineup_stats(away_starters)
        h_hier_off, h_hier_def = hier_engine.lineup_rating(home_starters)
        a_hier_off, a_hier_def = hier_engine.lineup_rating(away_starters)

        h_o_rd, h_d_rd = elo_tracker.lineup_uncertainty(home_starters)
        a_o_rd, a_d_rd = elo_tracker.lineup_uncertainty(away_starters)

        # Rolling xPPP
        if team_xppp_tracker is not None:
            h_roll_off, h_roll_def = team_xppp_tracker.get_rolling_xppp(home, current_season, gdate)
            a_roll_off, a_roll_def = team_xppp_tracker.get_rolling_xppp(away, current_season, gdate)
        else:
            h_roll_off = h_roll_def = a_roll_off = a_roll_def = DEFAULT_LEAGUE_XPPP

        # Pace
        pred_poss = pace_tracker.get_expected_pace(home, away)
        h_pace = pace_tracker.get_team_pace(home) if hasattr(pace_tracker, 'get_team_pace') else pred_poss
        a_pace = pace_tracker.get_team_pace(away) if hasattr(pace_tracker, 'get_team_pace') else pred_poss
        pace_diff = h_pace - a_pace
        pace_abs_diff = abs(pace_diff)

        # Rest days
        h_rest = min((gdate - last_game_date.get(home, gdate - pd.Timedelta(7))).days, 14)
        a_rest = min((gdate - last_game_date.get(away, gdate - pd.Timedelta(7))).days, 14)

        # Market odds (opening)
        market_spread, market_ml = get_odds(gdate, home, odds_dict) if odds_dict else (np.nan, np.nan)

        # Recent form
        h_recent = np.mean(team_recent_net[home]) if team_recent_net[home] else 0.0
        a_recent = np.mean(team_recent_net[away]) if team_recent_net[away] else 0.0
        recent_diff = h_recent - a_recent

        # --- SOS (strength of schedule) ---
        h_sos = np.mean([net for _, net in opponent_history[home]]) if opponent_history[home] else 0.0
        a_sos = np.mean([net for _, net in opponent_history[away]]) if opponent_history[away] else 0.0
        sos_diff = h_sos - a_sos

        # Season context
        if update_engines:
            h_new_starters = len(set(home_starters) - team_rosters_seen[home]) / max(len(home_starters), 1)
            a_new_starters = len(set(away_starters) - team_rosters_seen[away]) / max(len(away_starters), 1)
            days_since_start = (gdate - season_start_date).days
            h_season_phase = team_games_played[home] / 82.0
            a_season_phase = team_games_played[away] / 82.0
        else:
            h_new_starters = a_new_starters = 0.0
            days_since_start = 0
            h_season_phase = a_season_phase = 0.0

        # Build feature dictionary (including SOS)
        raw_margin = act_h - act_a
        rows.append({
            "GAME_ID": game_id,
            "game_date": gdate,
            "home_team": home,
            "away_team": away,
            "h_elo_off": ho_off,
            "h_elo_def": ho_def,
            "a_elo_off": ao_off,
            "a_elo_def": ao_def,
            "elo_diff_off": ho_off - ao_off,
            "elo_diff_def": ho_def - ao_def,
            "elo_net": (ho_off - ao_def) - (ao_off - ho_def),
            "h_hier_off": h_hier_off,
            "h_hier_def": h_hier_def,
            "a_hier_off": a_hier_off,
            "a_hier_def": a_hier_def,
            "hier_net": (h_hier_off - a_hier_def) - (a_hier_off - h_hier_def),
            "exp_poss": pred_poss,
            "h_rest": h_rest,
            "a_rest": a_rest,
            "h_b2b": int(h_rest <= 1),
            "a_b2b": int(a_rest <= 1),
            "is_altitude": int(home in ALTITUDE_TEAMS),
            "h_experience": ho_exp,
            "a_experience": ao_exp,
            "days_since_season_start": days_since_start,
            "h_season_phase": h_season_phase,
            "a_season_phase": a_season_phase,
            "h_new_starters": h_new_starters,
            "a_new_starters": a_new_starters,
            "market_spread": market_spread,
            "market_ml": market_ml,
            "actual_home": act_h,
            "actual_away": act_a,
            "actual_margin": raw_margin,
            "actual_margin_capped": np.clip(raw_margin, -20.0, 20.0),
            "actual_total": act_h + act_a,
            "home_win": int(act_h > act_a),
            "h_rating_uncertainty": h_o_rd + h_d_rd,
            "a_rating_uncertainty": a_o_rd + a_d_rd,
            "uncertainty_diff": (h_o_rd + h_d_rd) - (a_o_rd + a_d_rd),
            "h_roll_off_xppp": h_roll_off,
            "h_roll_def_xppp": h_roll_def,
            "a_roll_off_xppp": a_roll_off,
            "a_roll_def_xppp": a_roll_def,
            "roll_net_xppp": (h_roll_off - a_roll_def) - (a_roll_off - h_roll_def),
            "h_recent_net": h_recent,
            "a_recent_net": a_recent,
            "recent_diff": recent_diff,
            "pace_diff": pace_diff,
            "pace_abs_diff": pace_abs_diff,
            "pace_interaction": pace_diff * ((ho_off - ao_def) - (ao_off - ho_def)),
            # --- SOS features ---
            "h_sos": h_sos,
            "a_sos": a_sos,
            "sos_diff": sos_diff,
        })

        # Post‑game updates
        if update_engines:
            team_games_played[home] += 1
            team_games_played[away] += 1
            team_rosters_seen[home].update(home_starters)
            team_rosters_seen[away].update(away_starters)

            # Update recent form
            team_recent_net[home].append(act_h - act_a)
            team_recent_net[away].append(act_a - act_h)

            # Update SOS: store opponent net rating (off - def)
            # For home team, the opponent is away team; for away team, opponent is home team.
            opponent_history[home].append((away, ao_off - ao_def))
            opponent_history[away].append((home, ho_off - ho_def))

            seas_prog = len(rows) / max(total_games, 1)

            for _, row in group.iterrows():
                hp = _parse_player_string(row.get("HOME_players", ""))
                ap = _parse_player_string(row.get("AWAY_players", ""))
                p = float(row.get("possessions", 1))
                xh = float(row.get("home_xpts", 0))
                xa = float(row.get("away_xpts", 0))
                uh = row.get("home_usage", {}) or {}
                ua = row.get("away_usage", {}) or {}
                per = int(row.get("PERIOD", 1))
                ss = float(row.get("HOME_SCORE_START", 0))
                as_ = float(row.get("AWAY_SCORE_START", 0))
                se = float(row.get("HOME_SCORE_END", 0))
                ae = float(row.get("AWAY_SCORE_END", 0))
                rh = float(row.get("home_pts", 0))
                ra = float(row.get("away_pts", 0))

                elo_tracker.process_stint(
                    ids_A=hp, ids_B=ap, poss=p,
                    xpts_A=xh, xpts_B=xa,
                    usage_A=uh, usage_B=ua,
                    period=per,
                    start_A=ss, start_B=as_,
                    end_A=se, end_B=ae,
                    season_progress=seas_prog,
                )
                hier_engine.update(hp, ap, rh, ra, p)

            if team_xppp_tracker is not None:
                team_xppp_tracker.update(home, gdate, current_season, home_xppp_game, away_xppp_game)
                team_xppp_tracker.update(away, gdate, current_season, away_xppp_game, home_xppp_game)

            last_game_date[home] = gdate
            last_game_date[away] = gdate
            pace_tracker.update_pace(home, away, tot_poss)

            # Update lineup cache
            actual_first = group.iloc[0]
            actual_home = _parse_player_string(actual_first.get("HOME_players", ""))
            actual_away = _parse_player_string(actual_first.get("AWAY_players", ""))
            lineup_cache[home] = actual_home
            lineup_cache[away] = actual_away

    return pd.DataFrame(rows)

## Cell 10 — XGBoost Meta Score Model with Isotonic Calibration

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from optuna.pruners import MedianPruner
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, brier_score_loss
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import TimeSeriesSplit

# ═══════════════════════════════════════════════════════════════════════════
#   ADAPTIVE SHRINKING TUNER — with plateau and exploration jitter
# ═══════════════════════════════════════════════════════════════════════════

class AdaptiveParameterBounds:
    """
    Tracks the history of best Optuna values across seasons and produces
    progressively narrower search ranges centered on the running best.
    Includes plateau (stops shrinking after N seasons) and occasional jitter
    to force exploration and avoid premature convergence.
    """
    DECAY_RATE = 0.85              # slower decay (was 0.75)
    MIN_RANGE_FRACTION = 0.25      # keep at least 25% of original range (was 10%)
    PLATEAU_AFTER = 5              # stop shrinking after this many seasons
    INITIAL_SHRINK = 0.50          # after first season, use ±50% of original range
    EXPLORE_JITTER = 0.05          # random ±5% jitter around the center (applied 30% of the time)

    def __init__(self):
        self.history = {}          # {season_int: {"param_name": best_value, ...}}
        self.seasons_seen = 0

    def record_best(self, season: int, best_params: dict):
        self.history[season] = best_params.copy()
        self.seasons_seen += 1
        print(f"✅ AdaptiveParameterBounds: Recorded best params for season {season}. "
              f"Total seasons: {self.seasons_seen}.")

    def _shrink_factor(self) -> float:
        if self.seasons_seen == 0:
            return 1.0
        if self.seasons_seen >= self.PLATEAU_AFTER:
            return self.MIN_RANGE_FRACTION  # fixed floor, no further shrinking
        raw = self.INITIAL_SHRINK * (self.DECAY_RATE ** (self.seasons_seen - 1))
        return max(raw, self.MIN_RANGE_FRACTION)

    def _running_best(self) -> dict:
        if not self.history:
            return {}
        latest_season = max(self.history.keys())
        return self.history[latest_season]

    def suggest_bounds_float(self, param_name: str, original_low: float, original_high: float,
                             log: bool = False) -> tuple:
        best = self._running_best()
        shrink = self._shrink_factor()
        original_span = original_high - original_low

        if param_name not in best or self.seasons_seen == 0:
            return original_low, original_high

        center = best[param_name]
        # Apply jitter with 30% probability to force exploration
        if self.seasons_seen > 2 and np.random.random() < 0.3:
            center = center * (1 + np.random.uniform(-self.EXPLORE_JITTER, self.EXPLORE_JITTER))

        half_width = (original_span * shrink) / 2.0

        if log:
            import math
            log_center = math.log(center)
            log_low = math.log(original_low)
            log_high = math.log(original_high)
            log_span = log_high - log_low
            half_log = (log_span * shrink) / 2.0
            new_low = math.exp(max(log_low, log_center - half_log))
            new_high = math.exp(min(log_high, log_center + half_log))
        else:
            new_low  = max(original_low,  center - half_width)
            new_high = min(original_high, center + half_width)

        # Ensure non‑zero range
        if new_high <= new_low:
            new_low  = max(original_low,  center - original_span * 0.05)
            new_high = min(original_high, center + original_span * 0.05)

        return new_low, new_high

    def suggest_bounds_int(self, param_name: str, original_low: int, original_high: int) -> tuple:
        best = self._running_best()
        shrink = self._shrink_factor()
        original_span = original_high - original_low

        if param_name not in best or self.seasons_seen == 0:
            return original_low, original_high

        center = int(round(best[param_name]))
        # Apply jitter with 30% probability
        if self.seasons_seen > 2 and np.random.random() < 0.3:
            center = int(round(center * (1 + np.random.uniform(-self.EXPLORE_JITTER, self.EXPLORE_JITTER))))

        half_width = max(1, int(round((original_span * shrink) / 2.0)))

        new_low  = max(original_low,  center - half_width)
        new_high = min(original_high, center + half_width)

        if new_high <= new_low:
            new_low  = max(original_low,  center - 1)
            new_high = min(original_high, center + 1)

        return new_low, new_high

    def report(self):
        print(f"\n{'═'*60}")
        print(f"  📊 AdaptiveParameterBounds Report")
        print(f"  Seasons recorded: {self.seasons_seen}")
        print(f"  Current range shrink factor: {self._shrink_factor():.3f}x of original")
        if self.history:
            latest = self._running_best()
            print("  Current best params (running anchor):")
            for k, v in latest.items():
                print(f"    {k:30s}: {v:.5f}" if isinstance(v, float) else f"    {k:30s}: {v}")
        print(f"{'═'*60}\n")

# ──────────────────────────────────────────────────────────────────────────────
# SAFE FEATURE SET – no look‑ahead, no future data
# ──────────────────────────────────────────────────────────────────────────────

def engineer_interaction_features(df):
    """
    Adds explicit non‑linear interaction features to the feature DataFrame.
    This makes them available to both Ridge and XGBoost.
    """
    df_eng = df.copy()

    # Quality × Pace
    df_eng['elo_pace_interaction'] = df_eng['elo_net'] * df_eng['exp_poss']

    # Quality × Rest differential
    rest_diff = df_eng['h_rest'] - df_eng['a_rest']
    df_eng['elo_rest_interaction'] = df_eng['elo_net'] * rest_diff

    # Form (rolling xPPP net) × Pace
    df_eng['form_pace_interaction'] = df_eng['roll_net_xppp'] * df_eng['exp_poss']

    # Uncertainty‑adjusted quality
    total_uncertainty = df_eng['h_rating_uncertainty'] + df_eng['a_rating_uncertainty']
    df_eng['elo_uncertainty_adj'] = df_eng['elo_net'] / (total_uncertainty + 1e-6)

    return df_eng

# In Cell 10, update SAFE_FEATURE_COLS

SAFE_FEATURE_COLS = [
    "elo_net", "hier_net",
    "h_elo_off", "h_elo_def", "a_elo_off", "a_elo_def",
    "elo_diff_off", "elo_diff_def",
    "h_hier_off", "h_hier_def", "a_hier_off", "a_hier_def",
    "exp_poss", "h_rest", "a_rest", "h_b2b", "a_b2b",
    "is_altitude", "h_season_phase", "a_season_phase",
    "h_rating_uncertainty", "a_rating_uncertainty", "uncertainty_diff",
    "h_roll_off_xppp", "h_roll_def_xppp", "a_roll_off_xppp", "a_roll_def_xppp",
    "roll_net_xppp",
    "elo_pace_interaction", "elo_rest_interaction", "form_pace_interaction", "elo_uncertainty_adj",
    # New features
    "h_recent_net", "a_recent_net", "recent_diff",
    "pace_diff", "pace_abs_diff", "pace_interaction",
    "h_sos", "a_sos", "sos_diff"
]

# Default values (kept for compatibility)
DEFAULT_XGB_PARAMS = {
    'max_depth': 2,
    'learning_rate': 0.01,
    'n_estimators': 300,
    'min_child_weight': 100,
    'colsample_bytree': 0.4,
    'subsample': 0.5,
    'reg_alpha': 10.0,
    'reg_lambda': 10.0,
    'gamma': 5.0,
    'objective': 'reg:pseudohubererror',   # Huber loss for robustness
    'huber_slope': 1.0,
    'eval_metric': 'mae',
    'early_stopping_rounds': 50,
    'n_jobs': -1,
    'random_state': 42
}
DEFAULT_RIDGE_ALPHA = 50.0
DEFAULT_BLEND_WEIGHT = 0.70   # kept only for signature compatibility

# ──────────────────────────────────────────────────────────────────────────────
# STACKED META MODEL – Ridge + XGBoost Residuals with Early Stopping
# ──────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, HuberRegressor, LogisticRegression
from sklearn.ensemble import StackingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error, brier_score_loss
from sklearn.model_selection import TimeSeriesSplit
from catboost import CatBoostRegressor
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold

# ------------------------------------------------------------
# MetaScoreModel – Parallel Stacking with Huber meta‑learner
# ------------------------------------------------------------
class MetaScoreModel:
    def __init__(self, ridge_alpha=10.0, cb_params=None, huber_epsilon=1.0,
                 use_isotonic_calibration=False, cv_splitter=None):
        """
        Parameters
        ----------
        ridge_alpha : float
            Regularisation strength for Ridge base estimator.
        cb_params : dict or None
            Parameters for CatBoostRegressor (depth, iterations, etc.).
        huber_epsilon : float
            Epsilon parameter for the HuberRegressor meta‑learner.
            Controls the transition point from quadratic to linear loss.
        use_isotonic_calibration : bool
            If True, use IsotonicRegression for win‑prob calibration;
            otherwise, use LogisticRegression (Platt scaling).
        """
        self.ridge_alpha = ridge_alpha
        self.huber_epsilon = huber_epsilon
        self.use_isotonic = use_isotonic_calibration

        # Default CatBoost params (safe, low depth to avoid overfitting)
        default_cb = {
            'depth': 4,
            'iterations': 300,
            'learning_rate': 0.05,
            'l2_leaf_reg': 3.0,
            'loss_function': 'Huber:delta=1.5',
            'verbose': 0,
            'random_seed': 42
        }
        if cb_params is not None:
            default_cb.update(cb_params)
        self.cb_params = default_cb

        # Base estimators
        base_models = [
            ('ridge', Ridge(alpha=self.ridge_alpha)),
            ('catboost', CatBoostRegressor(**self.cb_params))
        ]

        # Final meta‑learner with Huber loss
        final_estimator = HuberRegressor(epsilon=self.huber_epsilon)

        # Stacking regressor with internal cross‑validation (prevents leakage)
        # Default to 5-fold CV using integer – safe and partition‑guaranteed
        if cv_splitter is None:
            cv_splitter = 5   # integer, uses KFold(5, shuffle=False)
        self.stack = StackingRegressor(
            estimators=base_models,
            final_estimator=final_estimator,
            cv=cv_splitter,
            n_jobs=-1,
            passthrough=False
        )

        # In tune_meta_model (when building the stack for tuning), do the same.

        self.scaler = StandardScaler()
        self.calibrator = None
        self.fitted = False
        self.features = SAFE_FEATURE_COLS.copy()

    def _get_features(self, X_df):
        """Extract and clean the safe feature set."""
        return X_df[self.features].fillna(0)

    def fit(self, X_df, y_home, y_away, calib_df=None):
        """
        Fit the stacking ensemble on the base set, then optionally calibrate
        win probabilities on a held‑out calibration set.
        """
        X = self._get_features(X_df)
        y_margin = y_home - y_away
        y_margin_capped = np.clip(y_margin, -20.0, 20.0)   # cap target for robustness

        # Scale and fit stack
        X_scaled = self.scaler.fit_transform(X)
        self.stack.fit(X_scaled, y_margin_capped)

        # --- Calibration (if calibration set provided) ---
        if calib_df is not None:
            raw_preds = self._predict_raw(calib_df)['pred_margin']
            raw_preds = raw_preds.reshape(-1, 1)
            y_win_cal = (calib_df['actual_home'] > calib_df['actual_away']).astype(int)

            if self.use_isotonic:
                self.calibrator = IsotonicRegression(out_of_bounds='clip')
                self.calibrator.fit(raw_preds.ravel(), y_win_cal)
                print(f"✅ Isotonic calibration fitted on {len(raw_preds)} samples.")
            else:
                # Platt scaling (logistic) – use strong L2 to avoid overfitting
                self.calibrator = LogisticRegression(C=0.1, solver='lbfgs', max_iter=100)
                self.calibrator.fit(raw_preds, y_win_cal)
                print(f"✅ Platt calibration (logistic) fitted on {len(raw_preds)} samples.")

        self.fitted = True
        print(f"✅ Stacked ensemble fitted with {len(self.stack.estimators_)} base models.")

    def _predict_raw(self, X_df):
        """Return raw predicted margin (before calibration)."""
        X = self._get_features(X_df)
        X_scaled = self.scaler.transform(X)
        margin = self.stack.predict(X_scaled)
        return {'pred_margin': margin}

    def predict(self, feat_dict):
        """
        Predict for a single game given a feature dictionary.
        Returns: dict with pred_home, pred_away, pred_margin, pred_total, win_prob.
        """
        if not self.fitted:
            raise RuntimeError("Model must be fitted before predicting.")

        X_df = pd.DataFrame([feat_dict], columns=self.features).fillna(0)
        raw = self._predict_raw(X_df)
        raw_margin = raw['pred_margin'][0]

        # Calibrate win probability
        if self.calibrator is not None:
            if self.use_isotonic:
                win_prob = self.calibrator.predict([raw_margin])[0]
            else:
                win_prob = self.calibrator.predict_proba(np.array([[raw_margin]]))[0, 1]
        else:
            # Fallback sigmoid (no calibration)
            win_prob = 1.0 / (1.0 + np.exp(-raw_margin / 12.0))
            win_prob = np.clip(win_prob, 0.01, 0.99)

        # Use a league‑average total to split margin into home/away scores
        league_avg_total = 225.0
        pred_home = (league_avg_total + raw_margin) / 2.0
        pred_away = (league_avg_total - raw_margin) / 2.0

        return {
            'pred_home': float(pred_home),
            'pred_away': float(pred_away),
            'pred_margin': float(raw_margin),
            'pred_total': float(pred_home + pred_away),
            'win_prob': float(win_prob)
        }


# ------------------------------------------------------------
# Hyperparameter Tuning for MetaScoreModel
# ------------------------------------------------------------
def tune_meta_model(train_features_df, n_trials=25, season=None,
                    bounds: 'AdaptiveParameterBounds' = None,
                    use_isotonic_calib=False):
    """
    Tune hyperparameters for the parallel stacking ensemble using
    TimeSeriesSplit with a 15‑game embargo gap.

    Tuned parameters:
      - ridge_alpha
      - cb_depth (2–7)
      - cb_iter (300–1200)
      - cb_lr (log‑scaled)
      - cb_l2 (log‑scaled)
      - huber_epsilon (0.5–5.0)

    Combined objective: validation MAE + Brier penalty + overfit penalty.
    """
    df = train_features_df.dropna(subset=['actual_margin']).reset_index(drop=True)
    X = df[SAFE_FEATURE_COLS].fillna(0)
    y_margin = df['actual_margin']
    y_home = df['actual_home']
    y_away = df['actual_away']

    b = bounds
    EMBARGO = 15

    def objective(trial):
        # ---- Ridge ----
        ridge_alpha = trial.suggest_float('ridge_alpha', 0.1, 100.0, log=True)

        # ---- CatBoost (wider ranges) ----
        cb_depth = trial.suggest_int('cb_depth', 2, 7)                 # widened
        cb_iter = trial.suggest_int('cb_iter', 300, 1200, step=100)    # widened
        cb_lr = trial.suggest_float('cb_lr', 0.01, 0.1, log=True)
        cb_l2 = trial.suggest_float('cb_l2', 1.0, 10.0, log=True)

        # ---- Huber epsilon (new) ----
        huber_epsilon = trial.suggest_float('huber_epsilon', 1.01, 4.0)

        # Build stack
        base_models = [
            ('ridge', Ridge(alpha=ridge_alpha)),
            ('catboost', CatBoostRegressor(
                depth=cb_depth,
                iterations=cb_iter,
                learning_rate=cb_lr,
                l2_leaf_reg=cb_l2,
                loss_function='Huber:delta=1.5',
                verbose=0,
                random_seed=42
            ))
        ]
        final_estimator = HuberRegressor(epsilon=huber_epsilon)   # tuned epsilon


        # Use TimeSeriesSplit for internal stacking as well
        # Use a simple integer for CV – KFold with no shuffle (preserves order, no leakage)
        stack = StackingRegressor(
            estimators=base_models,
            final_estimator=final_estimator,
            cv=3,                # <-- integer uses KFold(3, shuffle=False) – always partitions all samples
            n_jobs=-1,
            passthrough=False
        )
        scaler = StandardScaler()

        # Outer CV: TimeSeriesSplit with embargo
        tscv_outer = TimeSeriesSplit(n_splits=3)
        scores = []

        for train_idx, val_idx in tscv_outer.split(X):
            # Apply embargo: skip first 15 games of validation
            val_idx = val_idx[EMBARGO:]
            if len(val_idx) < 20:
                continue

            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            ym_tr, ym_val = y_margin.iloc[train_idx], y_margin.iloc[val_idx]

            X_tr_scaled = scaler.fit_transform(X_tr)
            X_val_scaled = scaler.transform(X_val)

            stack.fit(X_tr_scaled, ym_tr)

            train_preds = stack.predict(X_tr_scaled)
            val_preds = stack.predict(X_val_scaled)

            val_probs = 1.0 / (1.0 + np.exp(-val_preds / 12.0))
            val_probs = np.clip(val_probs, 0.01, 0.99)

            train_mae = mean_absolute_error(ym_tr, train_preds)
            val_mae = mean_absolute_error(ym_val, val_preds)
            brier = brier_score_loss((ym_val > 0).astype(int), val_probs)

            overfit_penalty = max(0, (train_mae - val_mae) - 1.0) * 2.0
            combined = val_mae + (5.0 * brier) + overfit_penalty
            scores.append(combined)

        return np.mean(scores) if scores else 9999.0


    print("⏳ Tuning Parallel Stacking Ensemble (Ridge + CatBoost + Huber) with adaptive bounds, Huber epsilon, and embargo...")
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=3, interval_steps=1)
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"✅ Tuning complete. Best combined score: {study.best_value:.4f}")
    best = study.best_params

    if b is not None and season is not None:
        b.record_best(season, best)
        b.report()

    # Return grouped parameters (including huber_epsilon)
    return {
        'ridge_alpha': best['ridge_alpha'],
        'huber_epsilon': best['huber_epsilon'],
        'cb_params': {
            'depth': best['cb_depth'],
            'iterations': best['cb_iter'],
            'learning_rate': best['cb_lr'],
            'l2_leaf_reg': best['cb_l2']
        }
    }

print("✅ Enhanced MetaScoreModel & Tuning Framework ready (Huber epsilon, wider CatBoost, robust calibration).")

print("✅ Enhanced MetaScoreModel & Tuning Framework ready (Huber, embargo, adaptive bounds).")

✅ Enhanced MetaScoreModel & Tuning Framework ready (Huber epsilon, wider CatBoost, robust calibration).
✅ Enhanced MetaScoreModel & Tuning Framework ready (Huber, embargo, adaptive bounds).


In [ ]:
# Cell 10b – Rolling Isotonic Calibrator (dynamic probability calibration)
from sklearn.isotonic import IsotonicRegression
from collections import deque

class RollingCalibrator:
    """
    Maintains a rolling window of (raw_prediction, actual_outcome) pairs.
    Fits an IsotonicRegression on the last `window_size` games.
    Predicts calibrated probability for new raw predictions.
    """
    def __init__(self, window_size=250, out_of_bounds='clip'):
        self.window_size = window_size
        self.out_of_bounds = out_of_bounds
        self.buffer = deque(maxlen=window_size)
        self.model = None

    def update(self, raw_prob, outcome):
        """Add a new calibration point (raw_prob in [0,1], outcome 0/1)."""
        self.buffer.append((raw_prob, outcome))
        if len(self.buffer) >= 20:   # refit only after enough samples
            self._fit()

    def _fit(self):
        if len(self.buffer) < 5:
            return
        X = np.array([p for p, _ in self.buffer]).reshape(-1, 1)
        y = np.array([o for _, o in self.buffer])
        iso = IsotonicRegression(out_of_bounds=self.out_of_bounds, increasing=True)
        iso.fit(X.ravel(), y)
        self.model = iso

    def predict(self, raw_prob):
        """Calibrate a single raw probability (clips to [0.01,0.99])."""
        if self.model is None or len(self.buffer) < 5:
            return np.clip(raw_prob, 0.01, 0.99)
        raw = np.clip(raw_prob, 0.0, 1.0)
        cal = self.model.predict([raw])[0]
        return np.clip(cal, 0.01, 0.99)

## Cell 11 — Optuna Tuning of Hierarchical Engine Weights

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

def tune_elo_tracker(stints_df, league_xppp, n_trials=20):
    """
    Optimized Elo Tuner using Chronological Time-Series Cross-Validation.
    Fits ratings on past historical seasons and evaluates strictly on the following season.
    """
    df = stints_df.copy()

    # Ensure strict chronological order
    if "game_date" in df.columns:
        # Before grouping by game, ensure strict chronological order
        df = df.sort_values(["game_date", "GAME_ID", "stint_id"]).reset_index(drop=True)

    # Build or ensure the 'season' grouping column exists
    if 'season' not in df.columns and 'game_date' in df.columns:
        df['season'] = pd.to_datetime(df['game_date']).dt.year + (pd.to_datetime(df['game_date']).dt.month >= 9).astype(int)

    def objective(trial):
        # Parameters
        k_off = trial.suggest_float("k_off", 0.2, 1)
        k_def = trial.suggest_float("k_def", 0.2, 1)
        elo_scaling = trial.suggest_int("elo_scaling", 500, 1000)
        home_boost = trial.suggest_float("home_boost", 0.0001, 0.007)
        offseason_reversion = trial.suggest_float("offseason_reversion", 0.1, 0.30)
        usage_floor = trial.suggest_float("usage_floor", 0.20, 0.275)
        assist_split = trial.suggest_float("assist_split", 0.75, 0.95)

        config = {
            "K_OFF": k_off, "K_DEF": k_def,
            "ELO_SCALING_FACTOR": elo_scaling,
            "HOME_PPP_BOOST": home_boost,
            "OFFSEASON_REVERSION": offseason_reversion,
            "USAGE_FLOOR": usage_floor,
            "assist_split": assist_split
        }

        seasons = sorted(df['season'].unique()) if 'season' in df.columns else []

        # Chronological Walk-Forward Cross Validation across Seasons
        if len(seasons) >= 2:
            fold_maes = []
            for i in range(1, len(seasons)):
                train_seasons = seasons[:i]
                val_season = seasons[i]

                # Re-initialize tracker from scratch for this fold
                tracker = PlayerRatingTracker(config=config, league_xppp=league_xppp)

                # 1. Warm-up / Train Tracker on past seasons (updates ratings only)
                train_df = df[df['season'].isin(train_seasons)]
                for _, row in train_df.iterrows():
                    hp = _parse_player_string(row["HOME_players"])
                    ap = _parse_player_string(row["AWAY_players"])
                    poss = float(row.get("possessions", 1))
                    if not (np.isfinite(poss) and poss >= 1):
                        continue
                    tracker.process_stint(
                        ids_A=hp, ids_B=ap, poss=poss,
                        xpts_A=float(row.get("home_xpts", 0)), xpts_B=float(row.get("away_xpts", 0)),
                        usage_A=row.get("home_usage", {}), usage_B=row.get("away_usage", {}),
                        period=int(row.get("PERIOD", 1)),
                        start_A=float(row.get("HOME_SCORE_START", 0)), start_B=float(row.get("AWAY_SCORE_START", 0)),
                        end_A=float(row.get("HOME_SCORE_END", 0)), end_B=float(row.get("AWAY_SCORE_END", 0)),
                        season_progress=0.5
                    )

                # 2. Evaluate Tracker strictly on the out-of-sample Next Season
                val_df = df[df['season'] == val_season]
                yt, yp = [], []
                for _, row in val_df.iterrows():
                    hp = _parse_player_string(row["HOME_players"])
                    ap = _parse_player_string(row["AWAY_players"])
                    poss = float(row.get("possessions", 1))
                    if not (np.isfinite(poss) and poss >= 1):
                        continue

                    # Maintain core structural consistency: update ratings then measure prediction alignment
                    ho_off, ho_def, _ = tracker.lineup_stats(hp)
                    ao_off, ao_def, _ = tracker.lineup_stats(ap)
                    pred_ppp_H = league_xppp + home_boost + (ho_off - ao_def) / elo_scaling
                    pred_ppp_A = league_xppp - home_boost + (ao_off - ho_def) / elo_scaling
                    tracker.process_stint(
                        ids_A=hp, ids_B=ap, poss=poss,
                        xpts_A=float(row.get("home_xpts", 0)), xpts_B=float(row.get("away_xpts", 0)),
                        usage_A=row.get("home_usage", {}), usage_B=row.get("away_usage", {}),
                        period=int(row.get("PERIOD", 1)),
                        start_A=float(row.get("HOME_SCORE_START", 0)), start_B=float(row.get("AWAY_SCORE_START", 0)),
                        end_A=float(row.get("HOME_SCORE_END", 0)), end_B=float(row.get("AWAY_SCORE_END", 0)),
                        season_progress=0.5
                    )



                    yt.extend([float(row.get("home_pts", 0)), float(row.get("away_pts", 0))])
                    yp.extend([pred_ppp_H * poss, pred_ppp_A * poss])

                arr_t, arr_p = np.array(yt), np.array(yp)
                mask = np.isfinite(arr_t) & np.isfinite(arr_p)
                mae = mean_absolute_error(arr_t[mask], arr_p[mask]) if mask.sum() > 10 else 9999.0
                fold_maes.append(mae)

            return np.mean(fold_maes)
        else:
            # Fallback to standard tracking loop if multi-season folds aren't available
            tracker = PlayerRatingTracker(config=config, league_xppp=league_xppp)
            yt, yp = [], []
            for _, row in df.iterrows():
                hp = _parse_player_string(row["HOME_players"])
                ap = _parse_player_string(row["AWAY_players"])
                poss = float(row.get("possessions", 1))
                if not (np.isfinite(poss) and poss >= 1):
                    continue
                tracker.process_stint(
                    ids_A=hp, ids_B=ap, poss=poss,
                    xpts_A=float(row.get("home_xpts", 0)), xpts_B=float(row.get("away_xpts", 0)),
                    usage_A=row.get("home_usage", {}), usage_B=row.get("away_usage", {}),
                    period=int(row.get("PERIOD", 1)),
                    start_A=float(row.get("HOME_SCORE_START", 0)), start_B=float(row.get("AWAY_SCORE_START", 0)),
                    end_A=float(row.get("HOME_SCORE_END", 0)), end_B=float(row.get("AWAY_SCORE_END", 0)),
                    season_progress=0.5
                )
                ho_off, ho_def, _ = tracker.lineup_stats(hp)
                ao_off, ao_def, _ = tracker.lineup_stats(ap)
                pred_ppp_H = league_xppp + home_boost + (ho_off - ao_off) / elo_scaling
                pred_ppp_A = league_xppp - home_boost + (ao_off - ho_def) / elo_scaling
                yt.extend([float(row.get("home_pts", 0)), float(row.get("away_pts", 0))])
                yp.extend([pred_ppp_H * poss, pred_ppp_A * poss])
            arr_t, arr_p = np.array(yt), np.array(yp)
            mask = np.isfinite(arr_t) & np.isfinite(arr_p)
            return mean_absolute_error(arr_t[mask], arr_p[mask]) if mask.sum() > 10 else 9999.0

    def callback(study, trial):
        print(f"\nTrial {trial.number}:")
        for key, value in trial.params.items():
            print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
        print(f"  CV MAE: {trial.value:.4f}")

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[callback])

    print("\n" + "="*50)
    print("✅ ELO TUNING COMPLETE")
    print(f"Best CV MAE: {study.best_value:.4f}")
    print("Best parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
    print("="*50)

    return study.best_params


def tune_hierarchical(stints_df, n_trials=30):
    """
    Optimized Hierarchical Tuner using Chronological Time-Series Cross-Validation.
    Fits ratings on past historical seasons and evaluates strictly on the following season.
    """
    df = stints_df.copy()
    if "game_date" in df.columns:
        # Before grouping by game, ensure strict chronological order
        df = df.sort_values(["game_date", "GAME_ID", "stint_id"]).reset_index(drop=True)

    if 'season' not in df.columns and 'game_date' in df.columns:
        df['season'] = pd.to_datetime(df['game_date']).dt.year + (pd.to_datetime(df['game_date']).dt.month >= 9).astype(int)

    def objective(trial):
        w1 = trial.suggest_float("w1", 0.1, 1)        # Player Offense
        w2 = trial.suggest_float("w2", 0.1, .5)        # Player Defense
        w3 = trial.suggest_float("w3", 0.1, .5)        # Lineup Offense
        w5 = trial.suggest_float("w5", 0.1, 1)        # Team Defense
        k_off = trial.suggest_float("k_off", 0.05, 0.9)  # Hierarchical Offense Update Speed
        k_def = trial.suggest_float("k_def", 0.05, 0.90)  # Hierarchical Defense Update Speed
        league_rtg = trial.suggest_float("league_rtg", 110.5, 114.0)

        total = w1 + w2 + w3 + w5
        if total <= 0:
            return 9999.0

        seasons = sorted(df['season'].unique()) if 'season' in df.columns else []

        # Chronological Walk-Forward Cross Validation across Seasons
        if len(seasons) >= 2:
            fold_maes = []
            for i in range(1, len(seasons)):
                train_seasons = seasons[:i]
                val_season = seasons[i]

                # Re-initialize engine from scratch for this fold
                engine = HierarchicalPossessionEngine(
                    w1/total, w2/total, w3/total, w5/total,
                    k_off=k_off, k_def=k_def, league_avg_rtg=league_rtg
                )

                # 1. Warm-up / Train Engine on past seasons (updates parameters only)
                train_df = df[df['season'].isin(train_seasons)]
                for _, row in train_df.iterrows():
                    poss = float(row.get("possessions", 1))
                    rh   = float(row.get("home_pts", 0))
                    ra   = float(row.get("away_pts", 0))
                    if not (np.isfinite(poss) and poss >= 1):
                        continue
                    hp = _parse_player_string(row["HOME_players"])
                    ap = _parse_player_string(row["AWAY_players"])
                    engine.update(hp, ap, rh, ra, poss)

                # 2. Evaluate Engine strictly on the out-of-sample Next Season
                val_df = df[df['season'] == val_season]
                yt, yp = [], []
                for _, row in val_df.iterrows():
                    poss = float(row.get("possessions", 1))
                    rh   = float(row.get("home_pts", 0))
                    ra   = float(row.get("away_pts", 0))
                    if not (np.isfinite(poss) and poss >= 1):
                        continue

                    hp = _parse_player_string(row["HOME_players"])
                    ap = _parse_player_string(row["AWAY_players"])

                    # Out-of-sample prediction before updating ratings
                    xh, xa, _, _ = engine.predict_pts(hp, ap, poss)

                    if np.isfinite(xh) and np.isfinite(xa):
                        yt.extend([rh, ra])
                        yp.extend([xh, xa])
                    engine.update(hp, ap, rh, ra, poss)

                mae = mean_absolute_error(yt, yp) if len(yt) > 10 else 9999.0
                fold_maes.append(mae)

            return np.mean(fold_maes)
        else:
            # Fallback
            engine = HierarchicalPossessionEngine(
                w1/total, w2/total, w3/total, w5/total,
                k_off=k_off, k_def=k_def, league_avg_rtg=league_rtg
            )
            yt, yp = [], []
            for _, row in df.iterrows():
                poss = float(row.get("possessions", 1))
                rh   = float(row.get("home_pts", 0))
                ra   = float(row.get("away_pts", 0))
                if not (np.isfinite(poss) and poss >= 1):
                    continue
                hp = _parse_player_string(row["HOME_players"])
                ap = _parse_player_string(row["AWAY_players"])
                xh, xa, _, _ = engine.predict_pts(hp, ap, poss)
                if np.isfinite(xh) and np.isfinite(xa):
                    yt.extend([rh, ra])
                    yp.extend([xh, xa])
                engine.update(hp, ap, rh, ra, poss)
            return mean_absolute_error(yt, yp) if len(yt) > 10 else 9999.0

    def callback(study, trial):
        print(f"\nTrial {trial.number}:")
        for key, value in trial.params.items():
            print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
        print(f"  CV MAE: {trial.value:.4f}")

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[callback])

    print("\n" + "="*50)
    print("✅ HIERARCHICAL TUNING COMPLETE")
    print(f"Best CV MAE: {study.best_value:.4f}")
    print("Best parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")
    print("="*50)

    best = study.best_params.copy()
    best['league_avg_rtg'] = best.pop('league_rtg')
    return best

print("✅ Tuning functions ready.")

✅ Tuning functions ready.


## Cell 12 — Walk-Forward Simulation & ATS Backtesting

In [ ]:
# Cell 11 – PaceTracker (updated)

import numpy as np
from collections import deque

class PaceTracker:
    def __init__(self, team_window=10, league_window=150):
        self.team_window = team_window
        self.league_window = league_window
        self.team_history = {}
        self.league_history = deque(maxlen=league_window)

    def get_expected_pace(self, home_team, away_team):
        h_pace = np.mean(self.team_history.get(home_team, [100])) if self.team_history.get(home_team) else 100.0
        a_pace = np.mean(self.team_history.get(away_team, [100])) if self.team_history.get(away_team) else 100.0
        lg_pace = np.mean(self.league_history) if self.league_history else 100.0
        raw_pace = (h_pace + a_pace) - lg_pace
        return max(85.0, raw_pace)

    def update_pace(self, home_team, away_team, game_possessions):
        if home_team not in self.team_history:
            self.team_history[home_team] = deque(maxlen=self.team_window)
        if away_team not in self.team_history:
            self.team_history[away_team] = deque(maxlen=self.team_window)
        self.team_history[home_team].append(game_possessions)
        self.team_history[away_team].append(game_possessions)
        self.league_history.append(game_possessions)

    def get_team_pace(self, team):
        """Return average possessions per game for a team (rolling window)."""
        hist = self.team_history.get(team)
        if hist and len(hist) > 0:
            return np.mean(hist)
        else:
            return 100.0  # league average fallback

In [ ]:
from sklearn.linear_model import LogisticRegression
from collections import deque

class RollingPlattCalibrator:
    """
    Rolling Platt scaling: fits a logistic regression on a rolling window
    of (raw_margin, outcome) to calibrate win probabilities.
    """
    def __init__(self, window_size=300, min_samples=20):
        self.window_size = window_size
        self.min_samples = min_samples
        self.buffer = deque(maxlen=window_size)   # stores (raw_margin, outcome)
        self.model = None

    def update(self, raw_margin, outcome):
        """Add a new calibration point (raw margin, 0/1 outcome)."""
        self.buffer.append((raw_margin, outcome))
        if len(self.buffer) >= self.min_samples:
            self._fit()

    def _fit(self):
        X = np.array([x for x, _ in self.buffer]).reshape(-1, 1)
        y = np.array([y for _, y in self.buffer])
        # Use strong L2 regularization to avoid overfitting
        clf = LogisticRegression(C=0.1, solver='lbfgs', max_iter=100)
        clf.fit(X, y)
        self.model = clf

    def predict(self, raw_margin):
        """Return calibrated win probability (clipped to [0.01,0.99])."""
        if self.model is None or len(self.buffer) < self.min_samples:
            # Fallback: sigmoid with scaling
            p = 1.0 / (1.0 + np.exp(-raw_margin / 12.0))
            return np.clip(p, 0.01, 0.99)
        prob = self.model.predict_proba(np.array([[raw_margin]]))[0, 1]
        return np.clip(prob, 0.01, 0.99)

In [ ]:
# Cell 12 – run_simulation (rewritten with SOS, proper Kelly, Platt rolling calibration)
from collections import defaultdict, deque
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def run_simulation(
    season_df,
    hier_engine,
    elo_tracker,
    meta_model,
    pace_tracker,
    odds_dict,
    calibrator=None,               # should be a RollingPlattCalibrator instance
    team_xppp_tracker=None,
    spread_calibrator=None,
    conformal=None,
    inactive_map: dict = None,
    sos_window: int = 10,          # number of games for SOS rolling average
):
    """
    Walk‑forward simulation for a single season.
    """
    if "game_date" in season_df.columns:
        season_df = season_df.sort_values(["game_date", "GAME_ID", "stint_id"]).reset_index(drop=True)

    results = []
    resid_hist = []
    last_date = {}

    season_start_date = None
    team_games_played = defaultdict(int)
    team_rosters_seen = defaultdict(set)
    lineup_cache = {}

    # Recent form deques (net rating)
    RECENT_WINDOW = 10
    team_recent_net = defaultdict(lambda: deque(maxlen=RECENT_WINDOW))

    # --- NEW: Strength of Schedule (opponent net rating history) ---
    opponent_history = defaultdict(lambda: deque(maxlen=sos_window))

    for game_id, group in tqdm(season_df.groupby("GAME_ID", sort=False), desc="Walk-forward simulation"):
        first_row = group.iloc[0]
        gdate = first_row.get("game_date", pd.NaT)
        home_team = first_row.get("home_team", "HOME")
        away_team = first_row.get("away_team", "AWAY")

        if isinstance(gdate, pd.Timestamp):
            current_season = gdate.year + (1 if gdate.month >= 9 else 0)
        else:
            current_season = 2025

        if season_start_date is None or (isinstance(gdate, pd.Timestamp) and gdate.month > 8 and (gdate - season_start_date).days > 150):
            season_start_date = gdate if isinstance(gdate, pd.Timestamp) else pd.Timestamp.now()
            team_games_played.clear()
            team_rosters_seen.clear()
            lineup_cache.clear()
            team_recent_net.clear()
            opponent_history.clear()   # also reset SOS at season boundary

        # Starting lineups
        if home_team in lineup_cache:
            home_starters = lineup_cache[home_team]
        else:
            first_stint = group.iloc[0]
            home_starters = _parse_player_string(first_stint.get("HOME_players", ""))
            lineup_cache[home_team] = home_starters

        if away_team in lineup_cache:
            away_starters = lineup_cache[away_team]
        else:
            first_stint = group.iloc[0]
            away_starters = _parse_player_string(first_stint.get("AWAY_players", ""))
            lineup_cache[away_team] = away_starters

        # Aggregate actual outcomes
        act_h = act_a = tot_poss = 0.0
        home_tot_poss = away_tot_poss = 0.0
        home_tot_xpts = away_tot_xpts = 0.0

        for _, row in group.iterrows():
            act_h += float(row.get("home_pts", 0))
            act_a += float(row.get("away_pts", 0))
            tot_poss += float(row.get("possessions", 1.0))
            home_tot_poss += float(row.get("home_poss", 0))
            away_tot_poss += float(row.get("away_poss", 0))
            home_tot_xpts += float(row.get("home_xpts", 0))
            away_tot_xpts += float(row.get("away_xpts", 0))

        if home_tot_poss == 0:
            home_tot_poss = tot_poss / 2
        if away_tot_poss == 0:
            away_tot_poss = tot_poss / 2
        if act_h == 0 and act_a == 0:
            act_h = group.get("HOME_FINAL_SCORE", pd.Series([110.0])).max()
            act_a = group.get("AWAY_FINAL_SCORE", pd.Series([107.0])).max()
        if tot_poss == 0:
            tot_poss = 100.0

        home_xppp_game = home_tot_xpts / home_tot_poss if home_tot_poss > 0 else DEFAULT_LEAGUE_XPPP
        away_xppp_game = away_tot_xpts / away_tot_poss if away_tot_poss > 0 else DEFAULT_LEAGUE_XPPP

        # Inactivity decay
        if isinstance(gdate, pd.Timestamp) and hasattr(elo_tracker, 'apply_inactivity_decay'):
            elo_tracker.apply_inactivity_decay(home_starters, gdate.date())
            elo_tracker.apply_inactivity_decay(away_starters, gdate.date())

        # Pre‑game ratings
        ho_off, ho_def, ho_exp = elo_tracker.lineup_stats(home_starters)
        ao_off, ao_def, ao_exp = elo_tracker.lineup_stats(away_starters)
        h_hoff, h_hdef = hier_engine.lineup_rating(home_starters)
        a_hoff, a_hdef = hier_engine.lineup_rating(away_starters)

        h_o_rd, h_d_rd = elo_tracker.lineup_uncertainty(home_starters)
        a_o_rd, a_d_rd = elo_tracker.lineup_uncertainty(away_starters)

        # Rolling xPPP
        if team_xppp_tracker is not None:
            h_roll_off, h_roll_def = team_xppp_tracker.get_rolling_xppp(home_team, current_season, gdate)
            a_roll_off, a_roll_def = team_xppp_tracker.get_rolling_xppp(away_team, current_season, gdate)
        else:
            h_roll_off = h_roll_def = a_roll_off = a_roll_def = DEFAULT_LEAGUE_XPPP

        # Pace
        pred_poss = pace_tracker.get_expected_pace(home_team, away_team)
        h_pace = pace_tracker.get_team_pace(home_team) if hasattr(pace_tracker, 'get_team_pace') else pred_poss
        a_pace = pace_tracker.get_team_pace(away_team) if hasattr(pace_tracker, 'get_team_pace') else pred_poss
        pace_diff = h_pace - a_pace
        pace_abs_diff = abs(pace_diff)

        # Rest
        h_rest = min((gdate - last_date.get(home_team, gdate - pd.Timedelta(7))).days, 14) if isinstance(gdate, pd.Timestamp) else 2
        a_rest = min((gdate - last_date.get(away_team, gdate - pd.Timedelta(7))).days, 14) if isinstance(gdate, pd.Timestamp) else 2

        # Roster turnover
        h_new_starters = len(set(home_starters) - team_rosters_seen[home_team]) / max(len(home_starters), 1)
        a_new_starters = len(set(away_starters) - team_rosters_seen[away_team]) / max(len(away_starters), 1)

        # Inactive rotation
        if inactive_map is not None:
            inactives = set(inactive_map.get(game_id, []))
            h_missing_rotation = 0.0
            a_missing_rotation = 0.0
        else:
            h_missing_rotation = a_missing_rotation = 0.0

        days_since_start = (gdate - season_start_date).days if isinstance(gdate, pd.Timestamp) else 50
        h_season_phase = team_games_played[home_team] / 82.0
        a_season_phase = team_games_played[away_team] / 82.0

        # Market odds (opening)
        market_spread, market_ml = get_odds(gdate, home_team, odds_dict) if odds_dict else (np.nan, np.nan)

        # Recent form
        h_recent = np.mean(team_recent_net[home_team]) if team_recent_net[home_team] else 0.0
        a_recent = np.mean(team_recent_net[away_team]) if team_recent_net[away_team] else 0.0
        recent_diff = h_recent - a_recent

        # --- NEW: Strength of Schedule (SOS) ---
        # Compute SOS from opponent history (net rating of past opponents)
        h_sos = np.mean([net for _, net in opponent_history[home_team]]) if opponent_history[home_team] else 0.0
        a_sos = np.mean([net for _, net in opponent_history[away_team]]) if opponent_history[away_team] else 0.0
        sos_diff = h_sos - a_sos

        # Build feature dict (including SOS)
        feat = {
            "h_elo_off": ho_off, "h_elo_def": ho_def,
            "a_elo_off": ao_off, "a_elo_def": ao_def,
            "elo_diff_off": ho_off - ao_off, "elo_diff_def": ho_def - ao_def,
            "elo_net": (ho_off - ao_def) - (ao_off - ho_def),
            "h_hier_off": h_hoff, "h_hier_def": h_hdef,
            "a_hier_off": a_hoff, "a_hier_def": a_hdef,
            "hier_net": (h_hoff - a_hdef) - (a_hoff - h_hdef),
            "exp_poss": pred_poss,
            "h_rest": h_rest, "a_rest": a_rest,
            "h_b2b": int(h_rest <= 1), "a_b2b": int(a_rest <= 1),
            "is_altitude": int(home_team in ALTITUDE_TEAMS),
            "h_experience": ho_exp, "a_experience": ao_exp,
            "days_since_season_start": days_since_start,
            "h_season_phase": h_season_phase,
            "a_season_phase": a_season_phase,
            "h_new_starters": h_new_starters,
            "a_new_starters": a_new_starters,
            "h_missing_rotation": h_missing_rotation,
            "a_missing_rotation": a_missing_rotation,
            "h_rating_uncertainty": h_o_rd + h_d_rd,
            "a_rating_uncertainty": a_o_rd + a_d_rd,
            "uncertainty_diff": (h_o_rd + h_d_rd) - (a_o_rd + a_d_rd),
            "h_roll_off_xppp": h_roll_off,
            "h_roll_def_xppp": h_roll_def,
            "a_roll_off_xppp": a_roll_off,
            "a_roll_def_xppp": a_roll_def,
            "roll_net_xppp": (h_roll_off - a_roll_def) - (a_roll_off - h_roll_def),
            "market_spread": market_spread,
            "market_ml": market_ml,
            "h_recent_net": h_recent,
            "a_recent_net": a_recent,
            "recent_diff": recent_diff,
            "pace_diff": pace_diff,
            "pace_abs_diff": pace_abs_diff,
            "pace_interaction": pace_diff * ((ho_off - ao_def) - (ao_off - ho_def)),
            # SOS features
            "h_sos": h_sos,
            "a_sos": a_sos,
            "sos_diff": sos_diff,
        }

        # Prediction
        if hasattr(meta_model, 'fitted') and meta_model.fitted:
            preds = meta_model.predict(feat)
            # --- Calibration: use raw margin to get win_prob (Platt) ---
            if calibrator is not None:
                preds["win_prob"] = calibrator.predict(preds["pred_margin"])
        else:
            preds = {"pred_home": 110, "pred_away": 110, "pred_margin": 0.0,
                     "pred_total": 220, "win_prob": 0.5}

        # Spread bias correction
        if spread_calibrator is not None:
            corrected_spread = spread_calibrator.correct(preds["pred_margin"])
        else:
            corrected_spread = preds["pred_margin"]

        # Conformal (placeholder)
        if conformal is not None:
            conf_lower, conf_upper, conf_width = conformal.predict_interval(corrected_spread, alpha=0.10)
        else:
            conf_lower, conf_upper, conf_width = corrected_spread - 12.0, corrected_spread + 12.0, 24.0

        # --- Proper Kelly Fraction (moneyline) ---
        kelly_fraction = 0.0
        if not pd.isna(market_ml) and "win_prob" in preds:
            dec_home = american_to_decimal(market_ml)
            dec_away = american_to_decimal(-market_ml)
            ev_home = preds["win_prob"] * dec_home - 1
            ev_away = (1 - preds["win_prob"]) * dec_away - 1

            if ev_home > 0.03 and ev_home > ev_away:
                b = dec_home - 1
                kelly = (preds["win_prob"] * b - (1 - preds["win_prob"])) / b
            elif ev_away > 0.03:
                b = dec_away - 1
                kelly = ((1 - preds["win_prob"]) * b - preds["win_prob"]) / b
            else:
                kelly = 0.0

            # Fractional Kelly (25%) and uncertainty penalty
            FRACTIONAL_K = 0.25
            kelly = max(0.0, min(1.0, kelly * FRACTIONAL_K))

            rating_uncertainty = (h_o_rd + h_d_rd + a_o_rd + a_d_rd) / 4.0
            uncertainty_penalty = max(0.0, 1.0 - (rating_uncertainty - 150) / 400)
            kelly_fraction = kelly * uncertainty_penalty
        else:
            kelly_fraction = 0.0

        # Betting analysis
        # ─── Betting analysis with explicit edge‑sign validation ──────────────
        if 'bet_analysis' in globals() and callable(bet_analysis):
            ba = bet_analysis(
                model_spread=corrected_spread,
                market_spread=market_spread,
                model_win_prob=preds.get("win_prob"),
                market_ml=market_ml,
                min_edge_pts=GOOD_BET_EDGE,
                min_ev=0.03,
                conf_width=conf_width
            )
            edge_pts   = ba.get("spread_edge_pts", 0.0)
            direction  = ba.get("spread_direction", "Pass")
            conf_score = ba.get("spread_confidence", 0)
            stars      = ba.get("spread_stars", "No market")
            ml_ev      = ba.get("ml_ev", np.nan)
            ml_dir     = ba.get("ml_direction", "Pass")

            # ── Safety net: ensure direction matches the sign of the edge ──
            # If edge_pts > 0 we must bet Home; if < 0, Away.
            # If inconsistent, discard the bet to avoid accidental negative‑EV wagers.
            if direction != "Pass":
                if (edge_pts > 0 and direction != "Home") or (edge_pts < 0 and direction != "Away"):
                    direction = "Pass"
                    edge_pts = 0.0
        else:
            edge_pts, direction, conf_score, stars = 0.0, "Pass", 0, "No market"
            ml_ev, ml_dir = np.nan, "Pass"

        # Store results
        results.append({
            "GAME_ID": game_id,
            "DATE": gdate.date() if hasattr(gdate, "date") else gdate,
            "HOME": home_team, "AWAY": away_team,
            "PRED_HOME": round(preds.get("pred_home", 0), 1),
            "PRED_AWAY": round(preds.get("pred_away", 0), 1),
            "PRED_SPREAD": round(corrected_spread, 2),
            "PRED_TOTAL": round(preds.get("pred_total", 0), 2),
            "WIN_PROB": round(preds.get("win_prob", 0), 3),
            "ACTUAL_HOME": act_h, "ACTUAL_AWAY": act_a,
            "ACTUAL_MARGIN": act_h - act_a,
            "MARKET_SPREAD": market_spread,
            "MARKET_ML": market_ml,
            "EDGE": round(edge_pts, 2),
            "DIRECTION": direction,
            "CONFIDENCE": conf_score,
            "STARS": stars,
            "ML_EV": ml_ev,
            "ML_DIRECTION": ml_dir,
            "CONF_LOWER": round(conf_lower, 2),
            "CONF_UPPER": round(conf_upper, 2),
            "CONF_WIDTH": round(conf_width, 2),
            "KELLY_FRACTION": round(kelly_fraction, 4),
        })

        # --- Post‑game updates ---
        seas_prog = 0.5
        for _, row in group.iterrows():
            hp = _parse_player_string(row.get("HOME_players", ""))
            ap = _parse_player_string(row.get("AWAY_players", ""))
            p = float(row.get("possessions", 1))
            if hasattr(elo_tracker, 'process_stint'):
                elo_tracker.process_stint(
                    ids_A=hp, ids_B=ap, poss=p,
                    xpts_A=float(row.get("home_xpts", 0)), xpts_B=float(row.get("away_xpts", 0)),
                    usage_A=row.get("home_usage", {}), usage_B=row.get("away_usage", {}),
                    period=int(row.get("PERIOD", 1)),
                    start_A=float(row.get("HOME_SCORE_START", 0)), start_B=float(row.get("AWAY_SCORE_START", 0)),
                    end_A=float(row.get("HOME_SCORE_END", 0)), end_B=float(row.get("AWAY_SCORE_END", 0)),
                    season_progress=seas_prog
                )
            if hasattr(hier_engine, 'update'):
                hier_engine.update(hp, ap, float(row.get("home_pts", 0)), float(row.get("away_pts", 0)), p)

        # Update calibrators (Platt uses raw margin)
        if calibrator is not None and "pred_margin" in preds:
            calibrator.update(preds["pred_margin"], 1 if act_h > act_a else 0)

        if spread_calibrator is not None:
            spread_calibrator.update(preds["pred_margin"], act_h - act_a)

        if team_xppp_tracker is not None:
            team_xppp_tracker.update(home_team, gdate, current_season, home_xppp_game, away_xppp_game)
            team_xppp_tracker.update(away_team, gdate, current_season, away_xppp_game, home_xppp_game)

        # Update recent form (after game)
        team_recent_net[home_team].append(act_h - act_a)
        team_recent_net[away_team].append(act_a - act_h)

        # --- Update SOS history: store opponent net rating ---
        # Use opponent's net rating (off - def) from pre-game stats as a proxy
        # For home team, the opponent is away team; for away team, opponent is home team.
        opponent_history[home_team].append((away_team, ao_off - ao_def))   # away net
        opponent_history[away_team].append((home_team, ho_off - ho_def))   # home net

        last_date[home_team] = gdate
        last_date[away_team] = gdate
        team_games_played[home_team] += 1
        team_games_played[away_team] += 1
        team_rosters_seen[home_team].update(home_starters)
        team_rosters_seen[away_team].update(away_starters)

        # Update lineup cache
        actual_first = group.iloc[0]
        actual_home = _parse_player_string(actual_first.get("HOME_players", ""))
        actual_away = _parse_player_string(actual_first.get("AWAY_players", ""))
        lineup_cache[home_team] = actual_home
        lineup_cache[away_team] = actual_away

        resid_hist.append(abs(corrected_spread - (act_h - act_a)))

        if hasattr(pace_tracker, 'update_pace'):
            pace_tracker.update_pace(home_team, away_team, tot_poss)

    out = pd.DataFrame(results)
    if not out.empty:
        out["SPREAD_ERR"] = (out["PRED_SPREAD"] - out["ACTUAL_MARGIN"]).abs()
        out["TOTAL_ERR"] = (out["PRED_TOTAL"] - (out["ACTUAL_HOME"] + out["ACTUAL_AWAY"])).abs()
    return out

## Cell 13 — Injury Helpers

In [ ]:
def fetch_injuries(team_abbr, date_str=None, use_real=True):
    """Return list of injured player names for a team."""
    if not use_real: return []
    try:
        from nbainjuries import injury
        inj_df = injury.get_injuries()
        col_team = next((c for c in inj_df.columns if "team" in c.lower()), None)
        col_name = next((c for c in inj_df.columns if "name" in c.lower() or "player" in c.lower()), None)
        if col_team and col_name:
            return list(inj_df[inj_df[col_team].str.upper() == team_abbr.upper()][col_name])
    except Exception as e:
        print(f"  ⚠️  Injury API error: {e}")
    return []

print("✅ Injury helper ready.")


✅ Injury helper ready.


## Cell 14 — Live / Future Game Predictor

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#   Cell 14 — Live / Future Game Predictor (Safe Feature Version)
# ═══════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np

def predict_game(home_full, away_full, game_date,
                 hier_engine, elo_tracker, meta_model, pace_tracker,
                 stints_df, game_outcomes_df,
                 live_market_spread=None, live_market_ml=None,
                 manual_home_out=None, manual_away_out=None,
                 use_real_injuries=True):
    """
    Predicts a future (or live) game using the trained pipeline.
    Uses only the safe, non‑leaky feature set.
    """
    if not meta_model.fitted:
        raise RuntimeError("MetaScoreModel must be fitted before predicting games.")

    home = TEAM_MAP.get(home_full, home_full)
    away = TEAM_MAP.get(away_full, away_full)

    manual_home_out = manual_home_out or []
    manual_away_out = manual_away_out or []

    # ── 1. Injury fetch & Roster Adjustment ─────────────────────────────
    api_h = fetch_injuries(home, use_real=use_real_injuries)
    api_a = fetch_injuries(away, use_real=use_real_injuries)
    inj_h = list(set(api_h + manual_home_out))
    inj_a = list(set(api_a + manual_away_out))
    out_h_ids = {name_to_id.get(n) for n in inj_h if name_to_id.get(n)}
    out_a_ids = {name_to_id.get(n) for n in inj_a if name_to_id.get(n)}

    def last_roster(abbr):
        tg = game_outcomes_df[
            (game_outcomes_df["home_team"] == abbr) |
            (game_outcomes_df["away_team"] == abbr)]
        if tg.empty:
            return []
        gid = tg.iloc[-1]["GAME_ID"]
        is_home = tg.iloc[-1]["home_team"] == abbr
        col = "HOME_players" if is_home else "AWAY_players"
        roster = set()
        for _, row in stints_df[stints_df["GAME_ID"] == gid].iterrows():
            roster.update(_parse_player_string(row[col]))
        return [str(p) for p in roster if p and str(p) != "nan"]

    h_roster = [p for p in last_roster(home) if int(p) not in out_h_ids]
    a_roster = [p for p in last_roster(away) if int(p) not in out_a_ids]

    if not h_roster or not a_roster:
        h_roster, a_roster = ["0"], ["0"]

    gdate = pd.Timestamp(game_date)
    elo_tracker.apply_inactivity_decay(h_roster + a_roster, gdate.date())

    # ── 2. Calculate Pre‑Game Ratings (Safe Features Only) ──────────────
    # Elo net rating
    first_stint = group.iloc[0]
    home_starters = _parse_player_string(first_stint["HOME_players"])
    away_starters = _parse_player_string(first_stint["AWAY_players"])

    # Now compute ratings on starters
    ho_off, ho_def, _ = elo_tracker.lineup_stats(home_starters)
    a_hoff, a_hdef = hier_engine.lineup_rating(away_starters)
    elo_net = (ho_off - ao_def) - (ao_off - ho_def)

    # Hierarchical net rating
    h_hoff, h_hdef = hier_engine.lineup_rating(set(int(p) for p in h_roster))
    a_hoff, a_hdef = hier_engine.lineup_rating(set(int(p) for p in a_roster))
    hier_net = (h_hoff - a_hdef) - (a_hoff - h_hdef)

    # Expected possessions (pace)
    pred_poss = pace_tracker.get_expected_pace(home, away)

    # Rest days and back‑to‑back indicators
    def get_rest_days(team_abbr, current_date):
        team_games = game_outcomes_df[
            (game_outcomes_df["home_team"] == team_abbr) |
            (game_outcomes_df["away_team"] == team_abbr)]
        if team_games.empty:
            return 7
        last_date = pd.to_datetime(team_games.iloc[-1]["game_date"])
        return min((current_date - last_date).days, 14)

    h_rest = get_rest_days(home, gdate)
    a_rest = get_rest_days(away, gdate)
    h_b2b = int(h_rest <= 1)
    a_b2b = int(a_rest <= 1)

    # Altitude advantage
    is_altitude = int(home in ALTITUDE_TEAMS)

    # ── 3. Build Feature Dictionary (exactly matching SAFE_FEATURE_COLS) ──
    feat = {
        "elo_net": elo_net,
        "hier_net": hier_net,
        "exp_poss": pred_poss,
        "h_rest": h_rest,
        "a_rest": a_rest,
        "h_b2b": h_b2b,
        "a_b2b": a_b2b,
        "is_altitude": is_altitude,
    }

    # Get prediction from meta model
    preds = meta_model.predict(feat)

    # Edge calculation (if live market spread provided)
    if live_market_spread is not None and 'bet_analysis' in globals():
        edge, direction, conf, stars = bet_analysis(preds["pred_margin"], live_market_spread, 12.0)
    else:
        edge, direction, conf, stars = (0.0, "N/A", 0, "N/A")

    # Format outputs
    model_line = (f"{home} -{abs(preds['pred_margin']):.1f}" if preds["pred_margin"] > 0
                  else f"{away} -{abs(preds['pred_margin']):.1f}")
    mkt_line = (f"{home} {live_market_spread:+.1f}" if live_market_spread else "No market line provided")

    # Print report
    print("\n" + "═"*58)
    print("  🏀  GAME PREDICTION REPORT")
    print("═"*58)
    print(f"  Matchup       : {away_full} @ {home_full}")
    print(f"  Date          : {gdate.date()}")
    print(f"  ──────────────────────────────────────────────────────")
    print(f"  Projected Score  : {home} {preds['pred_home']:.1f}  |  {away} {preds['pred_away']:.1f}")
    print(f"  Model Spread     : {model_line}")
    print(f"  Projected Total  : {preds['pred_total']:.1f}  (Est. Pace: {pred_poss:.1f})")
    print(f"  Win Prob (Home)  : {preds['win_prob']*100:.1f}%")
    print(f"  ──────────────────────────────────────────────────────")
    print(f"  Vegas Line       : {mkt_line}")
    print(f"  Edge             : {edge:+.2f} pts  →  {direction}")
    print(f"  Confidence       : {conf}/100  {stars}")
    print(f"  ──────────────────────────────────────────────────────")
    print(f"  Rest Advantage   : Home {h_rest} days | Away {a_rest} days")
    print(f"  Injuries (Home)  : {', '.join(inj_h) if inj_h else 'None'}")
    print(f"  Injuries (Away)  : {', '.join(inj_a) if inj_a else 'None'}")
    print("═"*58)

    return preds

print("✅ Live predictor ready (safe features only).")

✅ Live predictor ready (safe features only).


## Cell 15 — FULL PIPELINE (Run This Cell to Execute Everything)

In [62]:
# ─── Player name lookup ──────────────────────────────────────────────
names_dict = {}
name_to_id = {}
if PLAYER_LIST_PATH.exists():
    _names_df  = pd.read_csv(PLAYER_LIST_PATH, low_memory=False)
    names_dict = _names_df.set_index("person_id")["display_first_last"].to_dict()
    name_to_id = {v: k for k, v in names_dict.items()}
    print(f"✅ Loaded {len(names_dict):,} player names from nba_players_all.csv.")
else:
    print("⚠️ Player mapping CSV not found.")

# ─── Odds Loaders ────────────────────────────────────────────────────
def load_historical_odds(spread_path, ml_path, preferred_book="Pinnacle Sports"):
    odds_data = {}
    if spread_path.exists():
        df_sp = pd.read_csv(spread_path)
        if preferred_book in df_sp["book_name"].values:
            df_sp = df_sp[df_sp["book_name"] == preferred_book]
        else:
            df_sp = df_sp.drop_duplicates(subset=["game_id"], keep="last")
        for _, row in df_sp.iterrows():
            odds_data[int(row["game_id"])] = {"spread": float(row["spread1"]), "ml": np.nan}

    if ml_path.exists():
        df_ml = pd.read_csv(ml_path)
        if preferred_book in df_ml["book_name"].values:
            df_ml = df_ml[df_ml["book_name"] == preferred_book]
        else:
            df_ml = df_ml.drop_duplicates(subset=["game_id"], keep="last")
        for _, row in df_ml.iterrows():
            gid = int(row["game_id"])
            if gid not in odds_data: odds_data[gid] = {"spread": np.nan, "ml": np.nan}
            odds_data[gid]["ml"] = float(row["price1"])

    print(f"✅ Loaded historical odds for {len(odds_data)} games.")
    return odds_data

def load_modern_odds(csv_path, date_col="game_date", home_col="home_team",
                     spread_col="spread_home_points", ml_col="money_home_odds",
                     time_col="timestamp", game_start_hour=19):
    """
    Load odds and keep only the earliest available line per game.
    If time_col is provided, filter to rows before game start (assume 7 PM ET).
    """
    odds_dict = {}
    try:
        df = pd.read_csv(csv_path)
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        # If a timestamp column exists, convert and use it for sorting/filtering
        if time_col and time_col in df.columns:
            df['timestamp'] = pd.to_datetime(df[time_col], errors='coerce')
            # Assume games start at game_start_hour (19 = 7 PM ET)
            # Filter to odds recorded before game start (same day, hour < game_start_hour)
            # But we don't have game start time per game, so we can take the earliest timestamp per game.
            # Better: sort by timestamp and keep first per game.
            df = df.sort_values([date_col, 'timestamp'])
        else:
            df = df.sort_values([date_col])

        # Group by date and home team, take the first (earliest) row
        df['game_date_only'] = df[date_col].dt.date
        grouped = df.groupby(['game_date_only', home_col], as_index=False).first()

        for _, row in grouped.iterrows():
            dt_val = row['game_date_only']
            if pd.isna(dt_val):
                continue
            home = str(row[home_col]).strip()
            if home == "Los Angeles Lakers": home = "LA Lakers"
            if home == "Los Angeles Clippers": home = "LA Clippers"

            s_val = row[spread_col] if pd.notna(row.get(spread_col)) else np.nan
            m_val = row[ml_col] if pd.notna(row.get(ml_col)) else np.nan

            # Auto-correct sign
            if pd.notna(s_val) and pd.notna(m_val):
                if m_val < 0 and s_val > 0:
                    s_val = -abs(s_val)
                elif m_val > 0 and s_val < 0:
                    s_val = abs(s_val)

            odds_dict[(dt_val, home)] = {"spread": float(s_val), "ml": float(m_val)}

        print(f"✅ Loaded {len(odds_dict)} opening lines (earliest per game).")
    except Exception as e:
        print(f"⚠️ Error loading modern odds: {e}")
    return odds_dict

✅ Loaded 4,885 player names from nba_players_all.csv.


In [63]:
# ═════════════════════════════════════════════════════════════════════════════
#   CELL 2: REPORTING & DIAGNOSTIC FUNCTIONS
# ═════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")


def plot_model_diagnostics(final_results):
    if final_results.empty: return

    df = final_results.copy()
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df = df.sort_values("DATE").reset_index(drop=True)
    df["ROLLING_MAE"] = df["SPREAD_ERR"].rolling(window=20, min_periods=5).mean()

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.flatten()

    # 1. Spread Error Distribution
    axes[0].hist(df["SPREAD_ERR"].dropna(), bins=30, edgecolor="black", color="#4C72B0")
    axes[0].axvline(df["SPREAD_ERR"].mean(), color="red", linestyle="--", label=f"MAE={df['SPREAD_ERR'].mean():.2f}")
    axes[0].set_title("Spread Error Distribution")
    axes[0].legend()

    # 2. Predicted vs Actual Spread
    axes[1].scatter(df["ACTUAL_MARGIN"], df["PRED_SPREAD"], alpha=0.4, s=15, color="#55A868")
    lim = max(abs(df["ACTUAL_MARGIN"].max()), abs(df["PRED_SPREAD"].max())) + 5
    axes[1].plot([-lim, lim], [-lim, lim], "r--")
    axes[1].set_xlabel("Actual Margin")
    axes[1].set_ylabel("Pred Spread")
    axes[1].set_title("Pred vs Actual Spread")

    # 3. Probability Calibration Curve
    if "WIN_PROB" in df.columns:
        bins = np.linspace(0, 1, 11)
        wp   = df["WIN_PROB"].values
        act  = df["home_win"].values if "home_win" in df else ((df["ACTUAL_HOME"] > df["ACTUAL_AWAY"]).astype(int).values)
        bin_ids = np.digitize(wp, bins) - 1
        cal_x, cal_y = [], []
        for b in range(len(bins) - 1):
            mask = bin_ids == b
            if mask.sum() > 3:
                cal_x.append(wp[mask].mean()); cal_y.append(act[mask].mean())
        axes[2].plot(cal_x, cal_y, "o-", label="Model", color="#C44E52")
        axes[2].plot([0, 1], [0, 1], "r--", label="Perfect")
        axes[2].set_title("Win Prob Calibration")
        axes[2].legend()
        axes[2].set_xlabel("Predicted")
        axes[2].set_ylabel("Actual")

    # 4. Moving Average Error Consistency
    axes[3].plot(df["DATE"], df["ROLLING_MAE"], color="#8172B3", linewidth=2)
    axes[3].axhline(df["SPREAD_ERR"].mean(), color="red", linestyle="--", alpha=0.6)
    axes[3].set_title("Rolling 20-Game Spread MAE")
    axes[3].set_xlabel("Date")
    axes[3].set_ylabel("Rolling MAE (Points)")
    axes[3].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig("model_diagnostics.png", dpi=120, bbox_inches="tight")
    print("✅ Diagnostic charts saved cleanly to model_diagnostics.png")
    try: plt.show()
    except Exception: pass

In [ ]:
import copy
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

def run_multi_year_backtest_walkforward(
    stints_df: pd.DataFrame,
    odds_dict: dict = None,
    n_tuning_trials_elo: int = 10,
    n_tuning_trials_hier: int = 10,
    n_tuning_trials_meta: int = 5,
    rolling_window_size: int = 3,
) -> pd.DataFrame:
    """
    True walk‑forward backtest with NO data leakage.

    For each test season:
      1. Training = last `rolling_window_size` seasons (chronological).
      2. Tune Elo and Hierarchical on training seasons (nested CV).
      3. Split training games into 80% base / 20% calibration (chronological).
      4. Generate base features (engines updated sequentially).
      5. Generate calibration features using frozen copies of the engines.
      6. Tune meta‑model hyperparameters on base features only.
      7. Train final meta‑model on base features, then calibrate win prob on calibration set.
      8. Simulate test season with engines trained on all training data.
    """
    stints_df = stints_df.copy()
    stints_df["game_date"] = pd.to_datetime(stints_df["game_date"], errors="coerce")
    if "season" not in stints_df.columns:
        stints_df['season'] = stints_df['game_date'].dt.year + (stints_df['game_date'].dt.month >= 9).astype(int)

    all_seasons = sorted(stints_df['season'].dropna().unique())
    print(f"\n🚀 Walk‑forward over {len(all_seasons)} seasons: {all_seasons}")
    print(f"   Rolling window size = {rolling_window_size} season(s)")

    compiled_results = []

    for i, test_season in enumerate(all_seasons):
        print(f"\n{'='*65}")
        print(f"🚀 SIMULATING SEASON {int(test_season)-1}-{int(test_season)}")
        print('='*65)

        # ─── 1. Select training seasons (rolling window) ────────────────
        train_seasons = all_seasons[max(0, i - rolling_window_size):i]
        if not train_seasons:
            print("  [!] No previous seasons – skipping (no training data).")
            continue

        print(f"  Training on seasons: {train_seasons}")

        train_stints = stints_df[stints_df['season'].isin(train_seasons)].copy()
        test_stints = stints_df[stints_df['season'] == test_season].copy()

        # ─── 2. Tune Elo and Hierarchical on training data only ─────────
        print("  ⏳ Tuning Elo tracker on past data...")
        best_elo = tune_elo_tracker(train_stints, DEFAULT_LEAGUE_XPPP, n_trials=n_tuning_trials_elo)
        print("  ⏳ Tuning Hierarchical engine on past data...")
        best_hier = tune_hierarchical(train_stints, n_trials=n_tuning_trials_hier)

        # ─── 3. Split training games chronologically (80/20) ────────────
        game_dates = (
            train_stints[['game_date', 'GAME_ID']]
            .drop_duplicates('GAME_ID')
            .sort_values('game_date')
        )
        split_idx = int(len(game_dates) * 0.8)
        cutoff_date = game_dates.iloc[split_idx]['game_date']

        base_game_ids = game_dates.iloc[:split_idx]['GAME_ID'].values
        calib_game_ids = game_dates.iloc[split_idx:]['GAME_ID'].values

        base_stints = train_stints[train_stints['GAME_ID'].isin(base_game_ids)].copy()
        calib_stints = train_stints[train_stints['GAME_ID'].isin(calib_game_ids)].copy()
        print(f"  Base games: {len(base_game_ids)} | Calibration games: {len(calib_game_ids)}")

        # ─── 4. Generate BASE features (engines updated sequentially) ───
                # ─── 4. Generate BASE features ──────────────────────────────────────
        base_hier = HierarchicalPossessionEngine(**best_hier)
        base_elo = PlayerRatingTracker(config=best_elo, league_xppp=DEFAULT_LEAGUE_XPPP)
        base_pace = PaceTracker(team_window=10, league_window=100)

        base_features = generate_features(
            base_stints, base_hier, base_elo, base_pace,
            odds_dict=odds_dict,
            update_engines=True
        )
        base_features = engineer_interaction_features(base_features)   # <-- ADD THIS

        # ─── 5. Generate CALIBRATION features using FROZEN engine copies ──
        calib_hier = copy.deepcopy(base_hier)
        calib_elo = copy.deepcopy(base_elo)
        calib_pace = copy.deepcopy(base_pace)

        calib_features = generate_features(
            calib_stints, calib_hier, calib_elo, calib_pace,
            odds_dict=odds_dict,
            update_engines=True     # Engines evolve realistically
        )
        calib_features = engineer_interaction_features(calib_features) # <-- ADD THIS

        # ─── 6. Tune meta‑model hyperparameters on BASE features only ───
        print("  ⏳ Tuning MetaScoreModel hyperparameters...")
        print("  ⏳ Tuning MetaScoreModel hyperparameters...")
        best_params = tune_meta_model(
            base_features, n_trials=n_tuning_trials_meta
        )

        # Build the final stack with the best found parameters
        base_models = [
            ('ridge', Ridge(alpha=best_params['ridge_alpha'])),
            ('catboost', CatBoostRegressor(
                depth=best_params['cb_params']['depth'],
                iterations=best_params['cb_params']['iterations'],
                learning_rate=best_params['cb_params']['learning_rate'],
                l2_leaf_reg=best_params['cb_params']['l2_leaf_reg'],
                loss_function='Huber:delta=1.5',
                verbose=0,
                random_seed=42
            ))
        ]

        meta_model_regressor = HuberRegressor()
        stack = StackingRegressor(estimators=base_models, final_estimator=meta_model_regressor, cv=5)

        # Wrap it in your MetaScoreModel
        # Note: You will need to modify MetaScoreModel's __init__ slightly to accept a pre-built stack
        # or pass the parameters directly to it. The easiest way without changing MetaScoreModel is:

        meta_model = MetaScoreModel(
            ridge_alpha=best_params['ridge_alpha'],
            cb_params=best_params['cb_params'],
            huber_epsilon=best_params['huber_epsilon'],   # added
            use_isotonic_calibration=True
        )

        # Prepare targets
        y_home_base = base_features['actual_home']
        y_away_base = base_features['actual_away']

        # Fit the MetaScoreModel
        meta_model.fit(base_features, y_home_base, y_away_base, calib_df=calib_features)


        # ─── 8. Train engines on FULL training set (base + calib) ───────
        full_hier = HierarchicalPossessionEngine(**best_hier)
        full_elo = PlayerRatingTracker(config=best_elo, league_xppp=DEFAULT_LEAGUE_XPPP)
        full_pace = PaceTracker(team_window=10, league_window=100)
        _ = generate_features(
            train_stints, full_hier, full_elo, full_pace,
            odds_dict=odds_dict,
            update_engines=True
        )

        # ─── 9. Simulate test season ────────────────────────────────────
        sim_pace = PaceTracker(team_window=10, league_window=150)   # fresh for test
        # Apply offseason reversion before test season (realistic)
        if i > 0:
            full_hier.offseason_revert()
            full_elo.offseason_revert()

        # Rolling calibrator for online probability calibration during simulation
        rolling_calibrator = RollingPlattCalibrator(window_size=300, min_samples=20)
        spread_calibrator = SpreadCalibrator(window=50)

        results = run_simulation(
            season_df=test_stints,
            hier_engine=full_hier,
            elo_tracker=full_elo,
            meta_model=meta_model,
            pace_tracker=sim_pace,
            odds_dict=odds_dict,                # safe get_odds inside
            calibrator=rolling_calibrator
        )
        # In run_multi_year_backtest_walkforward, after the results = run_simulation(...) line:
        print(f"  Season {test_season} produced {len(results)} rows." if results is not None else "  Season returned None.")

        if results is not None and not results.empty:
            results['simulated_season_window'] = f"{int(test_season)-1}-{int(test_season)}"
            compiled_results.append(results)
        else:
            print(f"  ⚠️ No results for season {test_season}")

    if compiled_results:
        return pd.concat(compiled_results, ignore_index=True)
    return pd.DataFrame()

In [ ]:
import pandas as pd
import numpy as np

def print_interval_ats_report(results_df):
    """Calculates and prints Vegas ATS performance broken down by Edge intervals."""
    if results_df.empty:
        return

    # Filter for games where Vegas lines exist and we triggered an active bet
    df = results_df[results_df["MARKET_SPREAD"].notna() & (results_df["DIRECTION"] != "Pass")].copy()
    if df.empty:
        print("  [No Vegas Market lines found or zero active bets triggered]")
        return

    # Vegas Performance Scoring Logic
    home_covers = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) > 0
    away_covers = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) < 0
    pushes = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) == 0

    df["ATS_WIN"] = (((df["DIRECTION"] == "Home") & home_covers) | ((df["DIRECTION"] == "Away") & away_covers)).astype(int)
    df["ATS_LOSS"] = (((df["DIRECTION"] == "Home") & away_covers) | ((df["DIRECTION"] == "Away") & home_covers)).astype(int)
    df["ATS_PUSH"] = pushes.astype(int)

    # Bins for the discrete edge intervals
    bins = [-1.0, 2.999, 4.999, 7.999, float('inf')]
    labels = ["0.0 - 2.9", "3.0 - 4.9", "5.0 - 7.9", "8.0+"]
    df["EDGE_TIER"] = pd.cut(df["EDGE"], bins=bins, labels=labels, right=True)

    print("══════════════════════════════════════════════════════════════")
    print("           PREDICTION vs VEGAS (BY EDGE INTERVAL)           ")
    print("══════════════════════════════════════════════════════════════")

    for tier in labels:
        tier_df = df[df["EDGE_TIER"] == tier]
        total_bets = len(tier_df)

        if total_bets == 0:
            print(f" Edge {tier:>9} pts |   0 bets | No data")
            continue

        w, l, p = tier_df["ATS_WIN"].sum(), tier_df["ATS_LOSS"].sum(), tier_df["ATS_PUSH"].sum()
        win_pct = w / (w + l) if (w + l) > 0 else 0.0
        roi = (w * 0.909 - l) / total_bets if total_bets > 0 else 0.0

        print(f" Edge {tier:>9} pts | {total_bets:3d} bets | {w:2d}-{l:2d}-{p:1d} | {win_pct:5.1%} ATS | ROI: {roi:+.1%}")
    print("══════════════════════════════════════════════════════════════")


def generate_year_by_year_report(results_df):
    """Splits simulation results by NBA season and prints independent performance reports."""
    if results_df is None or results_df.empty:
        print("⚠️ Cannot generate report: results dataframe is empty.")
        return

    df = results_df.copy()
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")

    # Fallbacks to prevent crashes if error columns aren't calculated
    if "SPREAD_ERR" not in df.columns: df["SPREAD_ERR"] = 0.0
    if "TOTAL_ERR" not in df.columns: df["TOTAL_ERR"] = 0.0

    # Calculate NBA Season (Starts in Oct. If month >= 9, it is the start of a new season year)
    if "simulated_season_window" in df.columns:
        df["SEASON"] = df["simulated_season_window"]
    else:
        # Fallback: compute from DATE (for compatibility)
        df["SEASON"] = df["DATE"].apply(lambda d: f"{d.year}-{d.year+1}" if d.month >= 9 else f"{d.year-1}-{d.year}")


    seasons = sorted(df["SEASON"].dropna().unique())

    for season in seasons:
        season_df = df[df["SEASON"] == season]
        print(f"\n" + "═"*62)
        print(f"🚀 SEASON OVERVIEW: {season} ({len(season_df)} total tracked games)")
        print("═"*62)

        mae_sp = season_df["SPREAD_ERR"].mean()
        mae_to = season_df["TOTAL_ERR"].mean()
        print(f"  Spread MAE: {mae_sp:.2f} pts  |  Total MAE: {mae_to:.2f} pts\n")

        print_interval_ats_report(season_df)

    print(f"\n" + "█"*62)
    print(f"🏆 CUMULATIVE MULTI-YEAR TOTAL ({len(df)} total tracked games)")
    print("█"*62)

    mae_sp_tot = df["SPREAD_ERR"].mean()
    mae_to_tot = df["TOTAL_ERR"].mean()
    print(f"  Overall Spread MAE: {mae_sp_tot:.2f} pts  |  Total MAE: {mae_to_tot:.2f} pts\n")

    print_interval_ats_report(df)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ═════════════════════════════════════════════════════════════════════════════
#   0. HELPER FUNCTION: DEFINED HERE SO IT IS NEVER UNDEFINED
# ═════════════════════════════════════════════════════════════════════════════
def analyze_and_graph_seasons(results_df, title_prefix="Season", unit_size=10.0, min_bet_edge=2.0):
    if results_df is None or results_df.empty:
        return

    df = results_df[results_df["MARKET_SPREAD"].notna()].copy()
    if df.empty:
        print("  ⚠️ No Vegas odds matched in results. Financial report skipped.")
        return

    df["EDGE"] = abs(df["PRED_SPREAD"] + df["MARKET_SPREAD"])
    df["DIRECTION"] = np.where(df["PRED_SPREAD"] + df["MARKET_SPREAD"] > 0, "Home", "Away")

    # Compute ATS outcomes for all bets
    home_covers = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) > 0
    away_covers = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) < 0
    pushes = (df["ACTUAL_MARGIN"] + df["MARKET_SPREAD"]) == 0

    df["ATS_WIN"] = (((df["DIRECTION"] == "Home") & home_covers) |
                     ((df["DIRECTION"] == "Away") & away_covers)).astype(int)
    df["ATS_LOSS"] = (((df["DIRECTION"] == "Home") & away_covers) |
                      ((df["DIRECTION"] == "Away") & home_covers)).astype(int)
    df["ATS_PUSH"] = pushes.astype(int)

    # ─── SPREAD PROFITS FOR ALL BETS (not only active) ───
    # We'll compute profit for every bet, regardless of edge.
    # This lets you see what would have happened if you had bet on every game.
    df["SPREAD_PROFIT"] = 0.0
    win_payout = unit_size * (100 / 110)  # standard -110 odds

    # Mark active bets (edge > min_bet_edge) for later filtering
    df["ACTIVE_BET"] = df["EDGE"] > min_bet_edge

    # Compute profit for all bets
    mask_win = df["ATS_WIN"] == 1
    mask_loss = df["ATS_LOSS"] == 1
    df.loc[mask_win, "SPREAD_PROFIT"] = win_payout
    df.loc[mask_loss, "SPREAD_PROFIT"] = -unit_size

    # ─── MONEYLINE FINANCIALS (optional) ───
    df["ML_PROFIT"] = 0.0
    df["ML_BET"] = "Pass"
    if "MARKET_ML" in df.columns and "WIN_PROB" in df.columns:
        for idx, row in df.iterrows():
            ml = row["MARKET_ML"]
            if pd.isna(ml):
                continue
            vegas_prob_home = abs(ml) / (abs(ml) + 100) if ml < 0 else 100 / (ml + 100)
            model_prob_home = row["WIN_PROB"]
            away_ml = -ml if ml > 0 else abs(ml)

            if model_prob_home > (vegas_prob_home + 0.03):
                df.at[idx, "ML_BET"] = "Home"
                if row["ACTUAL_HOME"] > row["ACTUAL_AWAY"]:
                    df.at[idx, "ML_PROFIT"] = unit_size * (100 / abs(ml)) if ml < 0 else unit_size * (ml / 100)
                else:
                    df.at[idx, "ML_PROFIT"] = -unit_size
            elif (1.0 - model_prob_home) > ((1.0 - vegas_prob_home) + 0.03):
                df.at[idx, "ML_BET"] = "Away"
                if row["ACTUAL_AWAY"] > row["ACTUAL_HOME"]:
                    df.at[idx, "ML_PROFIT"] = unit_size * (100 / abs(away_ml)) if away_ml < 0 else unit_size * (away_ml / 100)
                else:
                    df.at[idx, "ML_PROFIT"] = -unit_size

    # ─── EDGE BINS ──────────────────────────────────────
    bins = [-1.0, 2.499, 4.999, 7.499, float('inf')]
    labels = ["0.0 - 2.5", "2.5 - 5.0", "5.0 - 7.5", "7.5+"]
    df["EDGE_TIER"] = pd.cut(df["EDGE"], bins=bins, labels=labels, right=True)

    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    if "simulated_season_window" in df.columns:
        df["SEASON"] = df["simulated_season_window"]
    else:
        # Compute NBA season (starts in Oct)
        df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
        df = df.dropna(subset=["DATE"])
        df["SEASON"] = df["DATE"].apply(
            lambda d: f"{d.year}-{d.year+1}" if d.month >= 9 else f"{d.year-1}-{d.year}"
        )

    for season, season_df in df.groupby("SEASON"):
        print(f"\n" + "═"*65)
        print(f"🏀 SEASON REPORT: {season}  |  Games Tracked: {len(season_df)}")
        print("═"*65)
        print(f"  Spread MAE: {season_df['SPREAD_ERR'].mean():.2f} pts  |  Total MAE: {season_df['TOTAL_ERR'].mean():.2f} pts\n")

        # ─── Print performance for each edge tier ──────────
        print(f"  SPREAD BETTING ($10 Unit) – All bets (including edge ≤ {min_bet_edge})")
        for tier in labels:
            tier_df = season_df[season_df["EDGE_TIER"] == tier]
            if len(tier_df) == 0:
                continue

            total_bets = len(tier_df)
            total_w = tier_df["ATS_WIN"].sum()
            total_l = tier_df["ATS_LOSS"].sum()
            total_p = tier_df["ATS_PUSH"].sum()
            total_win_pct = total_w / (total_w + total_l) if (total_w + total_l) > 0 else 0.0
            total_profit = tier_df["SPREAD_PROFIT"].sum()

            # Also count how many of these bets are "active" (edge > min_bet_edge)
            active_bets = tier_df[tier_df["ACTIVE_BET"]]
            active_cnt = len(active_bets)
            if active_cnt > 0:
                active_w = active_bets["ATS_WIN"].sum()
                active_l = active_bets["ATS_LOSS"].sum()
                active_p = active_bets["ATS_PUSH"].sum()
                active_win_pct = active_w / (active_w + active_l) if (active_w + active_l) > 0 else 0.0
                active_profit = active_bets["SPREAD_PROFIT"].sum()
                active_info = f" ({active_cnt} active: {active_w:2d}-{active_l:2d}-{active_p:1d} | {active_win_pct:5.1%} ATS | ${active_profit:+.2f})"
            else:
                active_info = ""

            print(f"  Edge {tier:>9} pts | {total_bets:3d} bets | {total_w:2d}-{total_l:2d}-{total_p:1d} | {total_win_pct:5.1%} ATS | Net: ${total_profit:+.2f}{active_info}")

        # ─── Moneyline summary (only active bets) ──────────
        ml_df = season_df[season_df["ML_BET"] != "Pass"]
        if not ml_df.empty:
            ml_wins = (ml_df["ML_PROFIT"] > 0).sum()
            ml_losses = (ml_df["ML_PROFIT"] < 0).sum()
            ml_profit = ml_df["ML_PROFIT"].sum()
            ml_win_pct = ml_wins / len(ml_df) if len(ml_df) > 0 else 0.0
            print(f"\n  MONEYLINE VALUE PLAYS ($10 Base Unit) – active only")
            print(f"  Plays: {len(ml_df):3d} | {ml_wins:2d} W - {ml_losses:2d} L | {ml_win_pct:5.1%} Hit | Net: ${ml_profit:+.2f}")

        # ─── Graphing (unchanged) ──────────────────────────
        season_df = season_df.sort_values("DATE").reset_index(drop=True)
        season_df["CUM_SPREAD"] = season_df["SPREAD_PROFIT"].cumsum()
        season_df["CUM_ML"] = season_df["ML_PROFIT"].cumsum()

        plt.figure(figsize=(10, 5))
        plt.plot(season_df.index, season_df["CUM_SPREAD"], label="Spread Profit ($10/bet)", color="#2ca02c", linewidth=2.5)
        plt.plot(season_df.index, season_df["CUM_ML"], label="Moneyline Profit ($10/bet)", color="#1f77b4", linewidth=2.5, alpha=0.8)
        plt.axhline(0, color='red', linestyle='--', alpha=0.6)
        graph_title = f"{title_prefix}_{season}_Equity_Curve"
        plt.title(f"{season} Equity Curve (All bets, min edge shown)", fontsize=13, fontweight="bold")
        plt.xlabel("Number of Games Played in Season")
        plt.ylabel("Cumulative Profit ($)")
        plt.legend(loc="upper left")
        plt.grid(alpha=0.3)
        plt.savefig(f"{graph_title.replace(' ', '_')}.png", dpi=120, bbox_inches="tight")
        plt.close()
        print(f"  ✅ Saved graph: {graph_title.replace(' ', '_')}.png")

    return df

In [ ]:
import gc
import pandas as pd

# ═════════════════════════════════════════════════════════════════════════════
#   MASTER PIPELINE EXECUTION (PHASE 1: ZERO-RAM DATA INGESTION)
# ═════════════════════════════════════════════════════════════════════════════

print("\n⏳ Loading modern Vegas lines...")
# Ensure your path variable matches (ODDS_CSV_PATH or MODERN_ODDS_PATH)
odds_path = MODERN_ODDS_PATH if 'MODERN_ODDS_PATH' in locals() else ODDS_CSV_PATH
modern_odds_dict = load_modern_odds(odds_path)

print("\n⏳ Ingesting and building complete historical timelines for backtest analytics...")
all_stints_frames = []

# 1. Process Historical Data (File by file to save RAM)
for year, path in V3_DATA_PATHS.items():
    if path.exists():
        try:
            print(f"  -> Processing {year}-{year+1} season: '{path.name}'")
            raw_df = pd.read_csv(path)

            # Run the mathematical pipeline for older V3 formats
            pbp_df = convert_v3_pbp(raw_df)
            pbp_df = preprocess_pbp(pbp_df, compute_xpoints=True)
            stints_df = build_stints(pbp_df)
            stints_df['season'] = int(year + 1)

            all_stints_frames.append(stints_df)

            # 🧹 Aggressive RAM Cleanup
            del raw_df, pbp_df, stints_df
            gc.collect()
        except Exception as e:
            print(f"  ⚠️ Error processing {path.name}: {e}")

# 2. Process Modern 2025-2026 Data (If it exists)
if 'PBP_2026_PATH' in locals() and PBP_2026_PATH.exists():
     try:
         print(f"  -> Processing live season track: '{PBP_2026_PATH.name}'")
         raw_df = pd.read_csv(PBP_2026_PATH)

         # ✅ THE FIX: Use convert_new_pbp for the modern combined-stats format!
         pbp_df = convert_new_pbp(raw_df, name_to_id)
         pbp_df = preprocess_pbp(pbp_df, compute_xpoints=True)
         stints_df = build_stints(pbp_df)
         stints_df['season'] = 2026

         all_stints_frames.append(stints_df)

         del raw_df, pbp_df, stints_df
         gc.collect()
     except Exception as e:
         print(f"  ⚠️ Error processing {PBP_2026_PATH.name}: {e}")

# 3. Combine all mathematically processed stints
if all_stints_frames:
    all_historical_stints = pd.concat(all_stints_frames, ignore_index=True)
    all_historical_stints["game_date"] = pd.to_datetime(all_historical_stints["game_date"], errors="coerce")
    all_historical_stints = all_historical_stints.sort_values(["game_date", "GAME_ID"]).reset_index(drop=True)

    total_games_found = len(all_historical_stints['GAME_ID'].unique())
    print(f"✅ Integrated Master Dataset Assembly Complete: {total_games_found:,} unique matches processed.")
else:
    raise ValueError("❌ Execution Blocked: No data processed. Check file paths.")


⏳ Loading modern Vegas lines...
✅ Loaded modern Vegas odds for 6069 games with auto-corrected signs.

⏳ Ingesting and building complete historical timelines for backtest analytics...
  -> Processing 2021-2022 season: 'events_2021_22_pbp_V3.csv'


KeyboardInterrupt: 

In [ ]:
print(len(all_historical_stints[all_historical_stints['season'] == 2025]))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#   MASTER WALK-FORWARD EVALUATION (Leakage‑Free, Safe Features)
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "█"*65)
print("📊 WALK-FORWARD EVALUATION (2021–2026) – LEAKAGE‑FREE PIPELINE")
print("   – Dynamic K‑factor in Glicko‑2 PlayerRatingTracker")
print("   – Platt‑calibrated win probabilities (logistic on raw margin)")
print("   – Meta model retrained each season using only past data")
print("   – Safe feature set (8 non‑leaky predictors)")
print("   – Huber loss (pseudo‑Huber) for robust spread regression")
print("   – Odds filtered by game date to prevent look‑ahead")
print("█"*65)

# Run the walk‑forward backtest across all seasons
all_results = run_multi_year_backtest_walkforward(
    stints_df=all_historical_stints,
    odds_dict=modern_odds_dict,                     # filtered inside get_odds
    n_tuning_trials_elo=25,
    n_tuning_trials_hier=25,
    n_tuning_trials_meta=10,
    rolling_window_size=3
)

if all_results is not None and not all_results.empty:
    # Generate detailed season reports and equity curves
    # min_bet_edge = 2.5 points (as recommended)
    analyze_and_graph_seasons(
        all_results,
        title_prefix="WalkForward_LeakageFree",
        unit_size=10.0,
        min_bet_edge=1.5
    )
    # Produce calibration diagnostics (reliability curve, MAE over time)
    plot_model_diagnostics(all_results)
    print("\n✅ Walk‑forward backtest completed successfully.")
else:
    print("⚠️ No results – check data availability and tuning settings.")


█████████████████████████████████████████████████████████████████
📊 WALK-FORWARD EVALUATION (2021–2026) – LEAKAGE‑FREE PIPELINE
   – Dynamic K‑factor in Glicko‑2 PlayerRatingTracker
   – Platt‑calibrated win probabilities (logistic on raw margin)
   – Meta model retrained each season using only past data
   – Safe feature set (8 non‑leaky predictors)
   – Huber loss (pseudo‑Huber) for robust spread regression
   – Odds filtered by game date to prevent look‑ahead
█████████████████████████████████████████████████████████████████

🚀 Walk‑forward over 5 seasons: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
   Rolling window size = 3 season(s)

🚀 SIMULATING SEASON 2021-2022
  [!] No previous seasons – skipping (no training data).

🚀 SIMULATING SEASON 2022-2023
  Training on seasons: [np.int64(2022)]
  ⏳ Tuning Elo tracker on past data...


  0%|          | 0/25 [00:00<?, ?it/s]


Trial 0:
  k_off: 0.4996
  k_def: 0.9606
  elo_scaling: 866
  home_boost: 0.0042
  offseason_reversion: 0.1312
  usage_floor: 0.2117
  assist_split: 0.7616
  CV MAE: 26.6087

Trial 1:
  k_off: 0.8929
  k_def: 0.6809
  elo_scaling: 854
  home_boost: 0.0002
  offseason_reversion: 0.2940
  usage_floor: 0.2624
  assist_split: 0.7925
  CV MAE: 26.6077

Trial 2:
  k_off: 0.3455
  k_def: 0.3467
  elo_scaling: 652
  home_boost: 0.0037
  offseason_reversion: 0.1864
  usage_floor: 0.2218
  assist_split: 0.8724
  CV MAE: 26.5742

Trial 3:
  k_off: 0.3116
  k_def: 0.4337
  elo_scaling: 683
  home_boost: 0.0032
  offseason_reversion: 0.2570
  usage_floor: 0.2150
  assist_split: 0.8528
  CV MAE: 26.5809

Trial 4:
  k_off: 0.6739
  k_def: 0.2372
  elo_scaling: 804
  home_boost: 0.0013
  offseason_reversion: 0.1130
  usage_floor: 0.2712
  assist_split: 0.9431
  CV MAE: 26.6012

Trial 5:
  k_off: 0.8467
  k_def: 0.4437
  elo_scaling: 548
  home_boost: 0.0048
  offseason_reversion: 0.1880
  usage_floor

  0%|          | 0/25 [00:00<?, ?it/s]


Trial 0:
  w1: 0.4371
  w2: 0.4803
  w3: 0.3928
  w5: 0.6388
  k_off: 0.1826
  k_def: 0.1826
  league_rtg: 110.7033
  CV MAE: 6.7215

Trial 1:
  w1: 0.8796
  w2: 0.3404
  w3: 0.3832
  w5: 0.1185
  k_off: 0.8744
  k_def: 0.7576
  league_rtg: 111.2432
  CV MAE: 6.1008

Trial 2:
  w1: 0.2636
  w2: 0.1734
  w3: 0.2217
  w5: 0.5723
  k_off: 0.4172
  k_def: 0.2975
  league_rtg: 112.6415
  CV MAE: 6.2395

Trial 3:
  w1: 0.2255
  w2: 0.2169
  w3: 0.2465
  w5: 0.5105
  k_off: 0.7174
  k_def: 0.2197
  league_rtg: 112.2998
  CV MAE: 6.1316

Trial 4:
  w1: 0.6332
  w2: 0.1186
  w3: 0.3430
  w5: 0.2535
  k_off: 0.1053
  k_def: 0.8566
  league_rtg: 113.8797
  CV MAE: 5.8907

Trial 5:
  w1: 0.8276
  w2: 0.2218
  w3: 0.1391
  w5: 0.7158
  k_off: 0.4241
  k_def: 0.1537
  league_rtg: 112.2331
  CV MAE: 5.9712

Trial 6:
  w1: 0.1309
  w2: 0.4637
  w3: 0.2035
  w5: 0.6963
  k_off: 0.3150
  k_def: 0.4921
  league_rtg: 112.4135
  CV MAE: 6.6096

Trial 7:
  w1: 0.2664
  w2: 0.4878
  w3: 0.4101
  w5: 0.9455


Building features:   0%|          | 0/1053 [00:00<?, ?it/s]

Building features:   0%|          | 0/264 [00:00<?, ?it/s]

  ⏳ Tuning MetaScoreModel hyperparameters...
  ⏳ Tuning MetaScoreModel hyperparameters...
⏳ Tuning Parallel Stacking Ensemble (Ridge + CatBoost + Huber) with adaptive bounds, Huber epsilon, and embargo...


  0%|          | 0/10 [00:00<?, ?it/s]

In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def display_edge_summary(results_df, plot=True):
    """
    Print a summary of ATS performance by edge bin.
    Bins: 0-1.5, 1.5-3, 3-4.5, 4.5-6, 6-7.5, 7.5-9, 9+
    """
    # Filter active bets (non‑Pass, market available)
    df = results_df[results_df['DIRECTION'] != 'Pass'].copy()
    if df.empty:
        print("No active bets found.")
        return

    if 'EDGE' not in df.columns:
        df['EDGE'] = abs(df['PRED_SPREAD'] + df['MARKET_SPREAD'])

    # Compute ATS win if not present
    if 'ATS_WIN' not in df.columns:
        home_covers = (df['ACTUAL_MARGIN'] + df['MARKET_SPREAD']) > 0
        away_covers = (df['ACTUAL_MARGIN'] + df['MARKET_SPREAD']) < 0
        df['ATS_WIN'] = ((df['DIRECTION'] == 'Home') & home_covers) | \
                        ((df['DIRECTION'] == 'Away') & away_covers)
        df['ATS_WIN'] = df['ATS_WIN'].astype(int)

    # Define 7 bins
    bins = [0, 1.5, 3.0, 4.5, 6.0, 7.5, 9.0, np.inf]
    labels = ['0-1.5', '1.5-3', '3-4.5', '4.5-6', '6-7.5', '7.5-9', '9+']
    df['edge_bin'] = pd.cut(df['EDGE'], bins=bins, labels=labels, right=False)

    total = len(df)
    summary = []
    for label in labels:
        bin_df = df[df['edge_bin'] == label]
        n = len(bin_df)
        if n == 0:
            win_pct = np.nan
            pct_bets = 0.0
        else:
            wins = bin_df['ATS_WIN'].sum()
            win_pct = 100 * wins / n
            pct_bets = 100 * n / total
        summary.append({
            'Bin': label,
            'Bets': n,
            '% of Total': pct_bets,
            'Win %': win_pct
        })

    summary_df = pd.DataFrame(summary)
    print("\n" + "="*60)
    print("ATS PERFORMANCE BY EDGE BIN")
    print("="*60)
    print(summary_df.to_string(index=False, formatters={
        '% of Total': '{:.1f}%'.format,
        'Win %': '{:.1f}%'.format
    }))
    print("="*60 + "\n")

    if plot:
        # Bar chart of win rates
        x = np.arange(len(labels))
        win_vals = summary_df['Win %'].fillna(0).values
        fig, ax = plt.subplots(figsize=(10,5))
        bars = ax.bar(x, win_vals, color='steelblue', edgecolor='black')
        ax.axhline(52.38, color='red', linestyle='--', label='Breakeven (52.38%)')
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45)
        ax.set_ylabel('ATS Win Rate (%)')
        ax.set_title('Win Rate by Predictive Edge Size')
        ax.legend()
        for i, bar in enumerate(bars):
            if win_vals[i] > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'{win_vals[i]:.1f}%', ha='center', va='bottom', fontsize=9)
        plt.tight_layout()
        plt.show()

    return summary_df

# ─── Call after backtest ─────────────────────────────
if 'all_results' in locals() and all_results is not None:
    display_edge_summary(all_results, plot=True)
else:
    print("No results – run backtest first.")


ATS PERFORMANCE BY EDGE BIN
  Bin  Bets % of Total Win %
0-1.5     0       0.0%   NaN
1.5-3    32       2.7% 68.8%
3-4.5    95       8.0% 58.9%
4.5-6    97       8.2% 56.7%
6-7.5    78       6.6% 60.3%
7.5-9    55       4.6% 56.4%
   9+    76       6.4% 59.2%



In [60]:
# 1. Check how many rows have a valid market spread for 2024-2025
df_2025 = all_results[all_results["simulated_season_window"] == "2024-2025"]
print("Rows for 2024-2025:", len(df_2025))
print("With market spread:", df_2025["MARKET_SPREAD"].notna().sum())

# 2. Check if any games in 2025 have market spread (you can also check overall)
print("Total rows with market spread in all_results:", all_results["MARKET_SPREAD"].notna().sum())

# 3. Check if the DATE column is correctly parsed
print("Sample DATE values for 2024-2025:")
print(df_2025["DATE"].head())

Rows for 2024-2025: 1314
With market spread: 354
Total rows with market spread in all_results: 1540
Sample DATE values for 2024-2025:
2626    2024-01-01
2627    2024-01-01
2628    2024-01-01
2629    2024-01-01
2630    2024-01-01
Name: DATE, dtype: object


In [61]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binom

def plot_edge_performance(results_df, min_bets_per_bin=10):
    """
    Create two plots showing ATS performance as a function of model edge.

    Parameters:
    -----------
    results_df : pd.DataFrame
        Output from run_simulation (must contain 'DIRECTION', 'PRED_SPREAD',
        'MARKET_SPREAD', 'ACTUAL_MARGIN', 'EDGE')
    min_bets_per_bin : int
        Minimum number of bets to display a bin (bins with fewer bets are skipped)
    """
    # Filter to active bets (non-Pass, market spread available)
    df = results_df[results_df['DIRECTION'] != 'Pass'].copy()
    if df.empty:
        print("No active bets found.")
        return

    # Compute ATS_WIN if not present
    if 'ATS_WIN' not in df.columns:
        print("Computing ATS_WIN from DIRECTION and actual margins...")
        # home covers if actual margin + market spread > 0
        home_covers = (df['ACTUAL_MARGIN'] + df['MARKET_SPREAD']) > 0
        away_covers = (df['ACTUAL_MARGIN'] + df['MARKET_SPREAD']) < 0
        df['ATS_WIN'] = ((df['DIRECTION'] == 'Home') & home_covers) | ((df['DIRECTION'] == 'Away') & away_covers)
        df['ATS_WIN'] = df['ATS_WIN'].astype(int)

    # Ensure EDGE column exists (if not, compute it)
    if 'EDGE' not in df.columns:
        print("Computing EDGE from PRED_SPREAD and MARKET_SPREAD...")
        df['EDGE'] = abs(df['PRED_SPREAD'] + df['MARKET_SPREAD'])

    # Create edge bins (0-2.5, 2.5-5, 5-7.5, 7.5+)
    bins = [0, 2.5, 5.0, 7.5, 100]
    labels = ['0-2.5', '2.5-5', '5-7.5', '7.5+']
    df['edge_bin'] = pd.cut(df['EDGE'], bins=bins, labels=labels, right=False)

    # Prepare aggregated data
    bin_stats = []
    for label in labels:
        mask = df['edge_bin'] == label
        n_bets = mask.sum()
        if n_bets < min_bets_per_bin:
            continue
        wins = df.loc[mask, 'ATS_WIN'].sum()
        loss = n_bets - wins
        win_rate = wins / n_bets if n_bets > 0 else 0
        # Wilson score interval for binomial proportion (95% CI)
        z = 1.96
        p_hat = win_rate
        n = n_bets
        if n > 0:
            lower = (p_hat + z**2/(2*n) - z * np.sqrt((p_hat*(1-p_hat) + z**2/(4*n))/n)) / (1 + z**2/n)
            upper = (p_hat + z**2/(2*n) + z * np.sqrt((p_hat*(1-p_hat) + z**2/(4*n))/n)) / (1 + z**2/n)
        else:
            lower, upper = 0, 0
        bin_stats.append({
            'bin': label,
            'n_bets': n_bets,
            'win_rate': win_rate,
            'lower': lower,
            'upper': upper,
            'avg_edge': df.loc[mask, 'EDGE'].mean()
        })

    bin_df = pd.DataFrame(bin_stats)

    # ---- Plot 1: Binned win rate with confidence intervals ----
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))

    if not bin_df.empty:
        x = np.arange(len(bin_df))
        width = 0.6
        bars = ax1.bar(x, bin_df['win_rate'], width,
                       yerr=[bin_df['win_rate']-bin_df['lower'], bin_df['upper']-bin_df['win_rate']],
                       capsize=5, color='steelblue', edgecolor='black')
        ax1.axhline(0.5, color='red', linestyle='--', label='Breakeven (50%)')
        ax1.set_xticks(x)
        ax1.set_xticklabels([f"{row['bin']}\n(n={row['n_bets']})" for _, row in bin_df.iterrows()])
        ax1.set_ylabel('ATS Win Rate')
        ax1.set_title('ATS Performance by Model Edge Bucket')
        ax1.legend()
        ax1.grid(axis='y', alpha=0.3)

        # Add win rate text on bars
        for i, row in bin_df.iterrows():
            ax1.text(i, row['win_rate'] + 0.02, f"{row['win_rate']:.1%}", ha='center', fontweight='bold')
    else:
        ax1.text(0.5, 0.5, "Insufficient data (no bin with enough bets)", ha='center', va='center')
        ax1.set_title('ATS Performance by Model Edge Bucket')

    # ---- Plot 2: Scatter of individual bets + rolling average ----
    if len(df) >= 10:
        # Sort by edge for rolling average
        df_sorted = df.sort_values('EDGE').reset_index(drop=True)
        df_sorted['rolling_win_rate'] = df_sorted['ATS_WIN'].rolling(window=50, min_periods=10).mean()

        # Jitter the y-values for better visibility
        np.random.seed(42)
        y_jitter = df_sorted['ATS_WIN'] + np.random.normal(0, 0.02, size=len(df_sorted))

        ax2.scatter(df_sorted['EDGE'], y_jitter, alpha=0.3, s=15, color='gray', label='Individual bet (jittered)')
        ax2.plot(df_sorted['EDGE'], df_sorted['rolling_win_rate'], color='darkred', linewidth=2,
                 label='Rolling 50‑bet win rate')
        ax2.axhline(0.5, color='red', linestyle='--', alpha=0.5)
        ax2.set_xlabel('Model Edge (points)')
        ax2.set_ylabel('ATS Outcome (1=Win, 0=Loss)')
        ax2.set_title('Individual Bet Performance vs. Edge (with rolling average)')
        ax2.legend(loc='upper left')
        ax2.grid(alpha=0.3)
    else:
        ax2.text(0.5, 0.5, f"Only {len(df)} active bets – need more for rolling average", ha='center', va='center')
        ax2.set_title('Individual Bet Performance vs. Edge')

    plt.tight_layout()
    plt.savefig('edge_performance_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Edge performance plot saved as 'edge_performance_analysis.png'")

    # Print summary table
    if not bin_df.empty:
        print("\n=== Edge Bucket Summary ===")
        print(bin_df[['bin', 'n_bets', 'win_rate', 'avg_edge']].to_string(index=False))
    else:
        print("\nNo edge buckets with sufficient bets (min_bets_per_bin = {})".format(min_bets_per_bin))

# Call the function after your walk-forward backtest
if 'all_results' in locals() and all_results is not None and not all_results.empty:
    plot_edge_performance(all_results)
else:
    print("No results available to plot.")

Computing ATS_WIN from DIRECTION and actual margins...
✅ Edge performance plot saved as 'edge_performance_analysis.png'

=== Edge Bucket Summary ===
  bin  n_bets  win_rate  avg_edge
2.5-5     156  0.583333  3.740769
5-7.5     146  0.609589  6.171918
 7.5+     131  0.580153  9.926794


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#   CELL 5: LIVE SYSTEM PREDICTION PREFERENCE
# ═════════════════════════════════════════════════════════════════════════════
import datetime

if 'meta_model' in locals() and meta_model.fitted:
    live_pace_tracker = PaceTracker()
    try:
        prediction = predict_game(
            home_full="Denver Nuggets",
            away_full="Los Angeles Lakers",
            game_date=datetime.datetime.today().strftime('%Y-%m-%d'),
            hier_engine=backtest_hier_engine,
            elo_tracker=backtest_elo_tracker,
            meta_model=meta_model,
            pace_tracker=live_pace_tracker,
            stints_df=all_historical_stints,
            game_outcomes_df=all_historical_stints,

            live_market_spread=-5.5,
            live_market_ml=-220,
            use_real_injuries=True
        )
    except NameError as e:
        print(f"⚠️ Live prediction function execution failed: {e}")
else:
    print("⚠️ Parameter Exception: Execution blocked. Run the pipeline cells above first to fit the MetaModel.")

In [ ]:
from nba_api.stats.endpoints import boxscoresummaryv2
import time
import pandas as pd
import ast
from tqdm.auto import tqdm
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# ─── Thread‑safe rate limiter ──────────────────────────────────────
class RateLimiter:
    def __init__(self, calls_per_second=1.0):
        self.lock = threading.Lock()
        self.interval = 1.0 / calls_per_second
        self.last_call = 0.0

    def wait(self):
        with self.lock:
            now = time.time()
            elapsed = now - self.last_call
            if elapsed < self.interval:
                time.sleep(self.interval - elapsed)
            self.last_call = time.time()

# ─── Fetch function (supports retries and rate limiting) ──────────
def fetch_inactive_for_game(game_id: str, rate_limiter: RateLimiter, retries: int = 2) -> list:
    """
    Fetch inactive player IDs with silent error handling.
    """
    for attempt in range(retries):
        try:
            rate_limiter.wait()
            summary = boxscoresummaryv2.BoxScoreSummaryV2(game_id=str(game_id))
            df = summary.inactive_players.get_data_frame()
            if df.empty:
                return []
            col = next((c for c in df.columns if c.lower() == 'player_id'), None)
            return df[col].tolist() if col else []
        except Exception:
            # By removing the print here, you stop the warnings from appearing.
            # We move to the next retry or return empty list if all retries fail.
            if attempt == retries - 1:
                return []
            time.sleep(1 * (attempt + 1))
    return []

def build_inactive_cache_parallel(
    stints_df: pd.DataFrame,
    cache_path=None,
    max_workers: int = 5,
    calls_per_second: float = 1.2,
):
    # 1. Handle Cache Loading
    if cache_path is not None:
        path = Path(cache_path)
        if path.exists():
            try:
                cached = pd.read_csv(path)
                cached['inactive_ids'] = cached['inactive_ids'].apply(ast.literal_eval)
                return cached.set_index('GAME_ID')['inactive_ids'].to_dict()
            except Exception as e:
                pass
                #print(f"⚠️ Failed to load cache: {e}")

    # 2. Setup Fetching Logic
    all_game_ids = stints_df['GAME_ID'].unique()
    #print(f"🔄 Fetching inactives for {len(all_game_ids)} games with {max_workers} workers...")

    rate_limiter = RateLimiter(calls_per_second=calls_per_second)
    results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_gid = {
            executor.submit(fetch_inactive_for_game, gid, rate_limiter): gid
            for gid in all_game_ids
        }

        with tqdm(total=len(all_game_ids), desc="Fetching inactive players") as pbar:
            for future in as_completed(future_to_gid):
                gid = future_to_gid[future]
                try:
                    # fetch_inactive_for_game is now silent
                    results[gid] = future.result(timeout=30)
                except Exception:
                    results[gid] = []
                pbar.update(1)

    # 3. Save to disk if path was provided
    if cache_path is not None:
        df_cache = pd.DataFrame(list(results.items()), columns=['GAME_ID', 'inactive_ids'])
        df_cache.to_csv(cache_path, index=False)
        print(f"✅ Built inactive cache for {len(results)} games.")

    return results


# ─── Usage ──────────────────────────────────────────────────────────
inactive_cache_path = None
#= ROOT / "inactive_players_cache.csv"
#inactive_map = build_inactive_cache_parallel(
#    all_historical_stints,
#    inactive_cache_path,
#    max_workers=10,          # increase if your network allows
#    calls_per_second=1.5    # safe limit)